# Limpieza de datos
En este notebook se limpia el dataset de e-commerce de Olist. Se parte de los CSV originales, se hace una copia con
errores puestos a propósito y se pasa por 15 reglas de limpieza hasta llegar a una capa CLEAN validada.

Para ejecutarlo: reiniciar el kernel y *Run All*. Las reglas van modificando las tablas `*_stg` una tras otra, así que
hay que correrlo en orden. Las transformaciones son todas SQL; Python solo arma y lanza las consultas.

| Tabla | Qué tiene |
|---|---|
| `*_raw` | los CSV tal cual, todo como texto |
| `*_stg` | copia de trabajo donde se aplican las reglas |
| `*_clean` | el resultado limpio, con tipos y marcas |
| `*_cuarentena` | lo que no se pudo arreglar, con el motivo |
| `quality_report` | métricas de cada regla, antes y después |

No todos los problemas se tratan igual. Si el arreglo es seguro (espacios, mayúsculas, un cero inicial perdido), se
corrige. Si el dato es imposible o falta algo crítico, el registro va a cuarentena: no se borra, se aparta con su motivo.
Y si es raro pero puede ser real, como un precio muy alto, se deja y se marca con un flag y una nota en `observaciones`.

Las reglas mantienen la numeración del enunciado, pero no se ejecutan en ese orden porque unas dependen de otras:

| Paso | Regla | Por qué va ahí |
|---|---|---|
| 1 | R14 Cadenas vacías | primero hay que convertir `''`, `'   '` y `'N/A'` en `NULL` |
| 2 | R03 Texto | para que `' DELIVERED'` y `'delivered'` sean lo mismo |
| 3 | R04 Tipos | sin fechas y números de verdad no se puede comparar nada |
| 4 | R01 Duplicados | con el texto ya normalizado aparecen también los que solo cambiaban de formato |
| 5 | R11 Códigos postales | R02 los usa para imputar ciudades |
| 6 | R02 Nulos | después de R14 y R04, que son las que generan `NULL` |
| 7 | R08 Categorías | usa la categoría que imputa R02 |
| 8 | R13 Métodos de pago | |
| 9 | R05 Fechas | necesita fechas con tipo |
| 10 | R10 Estados | |
| 11 | R06 Precios | necesita precios con tipo |
| 12 | R09 Integridad | cuando ya se sabe qué pedidos van a cuarentena, para arrastrar sus ítems y pagos |
| 13 | R07 Atípicos | sin precios inválidos que deformen los cuartiles |
| 14 | R12 Pagos | con ítems y pagos ya limpios |
| 15 | R15 Validación | sobre CLEAN |

## 0. Configuración

In [1]:
from pathlib import Path
import hashlib
import textwrap

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)

# Raíz del proyecto: el notebook vive en notebooks/, así que subimos un nivel si hace falta.
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_ORIGINAL = BASE_DIR / "data" / "original"    # CSV descargados de Kaggle: NUNCA se modifican
DIR_TRABAJO = BASE_DIR / "data" / "trabajo"      # copia con errores controlados (entrada del ETL)
DIR_PROCESSED = BASE_DIR / "data" / "processed"  # salida limpia en Parquet
DIR_REPORTS = BASE_DIR / "reports"
DIR_SQL = BASE_DIR / "sql"
for carpeta in (DIR_TRABAJO, DIR_PROCESSED, DIR_REPORTS, DIR_SQL):
    carpeta.mkdir(parents=True, exist_ok=True)

DB_PATH = BASE_DIR / "ecommerce.duckdb"
con = duckdb.connect(str(DB_PATH))

ARCHIVOS = {
    "pedidos": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "clientes": "olist_customers_dataset.csv",
    "pagos": "olist_order_payments_dataset.csv",
    "productos": "olist_products_dataset.csv",
}
TABLAS = list(ARCHIVOS)

print("Raíz del proyecto:", BASE_DIR)
print("Base de datos    :", DB_PATH)
print("DuckDB", duckdb.__version__)

Raíz del proyecto: d:\Users\luism\Documents\Personal\Uni\Maestria\analisis -de-datos\proyecto-inicial
Base de datos    : d:\Users\luism\Documents\Personal\Uni\Maestria\analisis -de-datos\proyecto-inicial\ecommerce.duckdb
DuckDB 2.0.0


### 0.1 Funciones de apoyo

- `run(sql)` ejecuta SQL que cambia datos y lo guarda en la lista `PIPELINE`. Esa lista se usa después para exportar el
  pipeline a `sql/pipeline_etl.sql` y para repetirlo en la prueba de idempotencia.
- `ver(sql)` es para consultas de solo lectura y devuelve un DataFrame.
- `marcar_cuarentena` y `marcar_observacion` aplican las dos políticas que no son corregir.

In [2]:
PIPELINE = []

def run(sql):
    """Ejecuta SQL que crea o modifica datos y lo registra en el pipeline."""
    sql = textwrap.dedent(sql).strip()
    if sql not in PIPELINE:
        PIPELINE.append(sql)
    return con.execute(sql)

def ver(sql):
    """Consulta de solo lectura; devuelve un DataFrame."""
    return con.execute(textwrap.dedent(sql)).df()

def columnas(tabla):
    """Columnas de una tabla, en su orden original."""
    filas = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_name = ? ORDER BY ordinal_position", [tabla]).fetchall()
    return [f[0] for f in filas]

def cualquiera(plantilla, cols):
    """Une con OR una condición aplicada a varias columnas: cualquiera('{c} IS NULL', ['a','b'])."""
    return " OR ".join(plantilla.format(c=f'"{c}"') for c in cols)

def lista_sql(valores):
    return "(" + ", ".join(f"'{v}'" for v in valores) + ")"

def marcar_cuarentena(tabla, motivo, condicion):
    """Agrega un motivo de cuarentena (acumulativo) a los registros que cumplen la condición."""
    n = run(f"""
        UPDATE {tabla}
        SET motivo_cuarentena = concat_ws('; ', motivo_cuarentena, '{motivo}')
        WHERE ({condicion})
          AND coalesce(motivo_cuarentena, '') NOT LIKE '%{motivo}%'
    """).fetchone()[0]
    print(f"  cuarentena  {tabla:<14}{n:>7}  {motivo}")
    return n

def marcar_observacion(tabla, flag, texto, condicion):
    """Conserva el registro pero lo marca: columna booleana `flag` + nota en `observaciones`."""
    run(f"ALTER TABLE {tabla} ADD COLUMN IF NOT EXISTS {flag} BOOLEAN DEFAULT false")
    n = run(f"""
        UPDATE {tabla}
        SET {flag} = true,
            observaciones = concat_ws('; ', observaciones, '{texto}')
        WHERE ({condicion})
          AND coalesce(observaciones, '') NOT LIKE '%{texto}%'
    """).fetchone()[0]
    print(f"  marcado     {tabla:<14}{n:>7}  {texto}")
    return n

### 0.2 Macros SQL

Macros de DuckDB que usan varias reglas. Aceptan tanto texto como valores ya convertidos, y por eso la misma métrica
se puede correr sobre RAW y sobre CLEAN.

In [3]:
TEXTOS_NULOS = "('NULL', 'N/A', 'NA', 'NONE', 'NAN', '-')"

run(f"""
-- R14: TRUE si el valor existe pero en realidad no contiene un dato ('', '   ', 'N/A', 'null'...)
CREATE OR REPLACE MACRO es_vacio(x) AS
    x IS NOT NULL
    AND (NULLIF(trim(CAST(x AS VARCHAR)), '') IS NULL
         OR upper(trim(CAST(x AS VARCHAR))) IN {TEXTOS_NULOS});

CREATE OR REPLACE MACRO limpiar_vacio(x) AS
    CASE WHEN es_vacio(x) THEN NULL ELSE x END;

-- R03: sin espacios sobrantes, en minúsculas y sin tildes
CREATE OR REPLACE MACRO normalizar_texto(x) AS
    lower(strip_accents(regexp_replace(trim(x), '[[:space:]]+', ' ', 'g')));

-- R04: texto -> número (acepta coma decimal) y texto -> timestamp (acepta dd/mm/aaaa). NULL si no se puede.
CREATE OR REPLACE MACRO a_numero(x) AS
    TRY_CAST(replace(trim(CAST(x AS VARCHAR)), ',', '.') AS DECIMAL(12, 2));

CREATE OR REPLACE MACRO a_fecha(x) AS
    TRY_STRPTIME(trim(CAST(x AS VARCHAR)),
                 ['%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%d/%m/%Y %H:%M:%S', '%d/%m/%Y']);
""")
print("Macros creadas")

Macros creadas


### 0.3 Parámetros del pipeline

In [4]:
# R01: claves de negocio. Un pedido puede tener varios ítems: la clave de items es (order_id, order_item_id).
CLAVES_NEGOCIO = {
    "pedidos": ["order_id"],
    "items": ["order_id", "order_item_id"],
    "clientes": ["customer_id"],
    "pagos": ["order_id", "payment_sequential"],
    "productos": ["product_id"],
}

# R02 / R04: campos críticos. Sin ellos el registro no se puede usar -> cuarentena.
CRITICAS = {
    "pedidos": ["order_id", "customer_id", "order_purchase_timestamp"],
    "items": ["order_id", "order_item_id", "product_id", "price", "freight_value"],
    "clientes": ["customer_id"],
    "pagos": ["order_id", "payment_sequential", "payment_value"],
    "productos": ["product_id"],
}

# R04: tipos destino. customer_zip_code_prefix se queda como TEXTO a propósito (R11: ceros iniciales).
COLS_FECHA_PEDIDOS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                      "order_delivered_customer_date", "order_estimated_delivery_date"]
CONVERSIONES = {
    "pedidos": {c: "TIMESTAMP" for c in COLS_FECHA_PEDIDOS},
    "items": {"order_item_id": "INTEGER", "shipping_limit_date": "TIMESTAMP",
              "price": "DECIMAL(10,2)", "freight_value": "DECIMAL(10,2)"},
    "pagos": {"payment_sequential": "INTEGER", "payment_installments": "INTEGER",
              "payment_value": "DECIMAL(10,2)"},
    "productos": {c: "INTEGER" for c in [
        "product_name_lenght", "product_description_lenght", "product_photos_qty",
        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]},
}

def conversion(col, tipo):
    """Expresión SQL que convierte una columna al tipo destino; NULL si el valor no es convertible."""
    if tipo == "TIMESTAMP":
        return f'a_fecha("{col}")'
    return f'TRY_CAST(a_numero("{col}") AS {tipo})'

ESTADOS_VALIDOS = ["created", "approved", "invoiced", "processing", "shipped", "delivered", "canceled", "unavailable"]
METODOS_VALIDOS = ["credit_card", "boleto", "voucher", "debit_card"]
TOLERANCIA_PAGO = 1.00   # BRL. Diferencia admitida entre lo pagado y ítems + flete (redondeo de cuotas).

### 0.4 Tabla `quality_report` (bonificación)

Cada regla define su métrica una sola vez, como una plantilla SQL con `{capa}`. La plantilla se corre sobre `*_raw`
(antes) y sobre `*_clean` (después), así las dos cifras salen de la misma consulta.

In [5]:
run("""
CREATE OR REPLACE TABLE quality_report (
    nombre_regla        VARCHAR,
    tabla_evaluada      VARCHAR,
    etapa               VARCHAR,     -- 'antes' = capa RAW, 'despues' = capa CLEAN
    registros_evaluados BIGINT,
    registros_invalidos BIGINT,
    porcentaje_error    DOUBLE,
    politica            VARCHAR,
    fecha_ejecucion     TIMESTAMP
)
""")

REGLAS = {
    "R01": ("Duplicados", "Corregir: se conserva un registro por clave de negocio (el más completo)"),
    "R02": ("Valores nulos", "Cuarentena si falta un campo crítico; imputar ciudad y categoría; conservar el resto"),
    "R03": ("Estandarización de texto", "Corregir: TRIM, minúsculas (UF en mayúsculas) y sin tildes"),
    "R04": ("Tipos de datos", "Convertir; valor crítico no convertible -> cuarentena; no crítico -> NULL"),
    "R05": ("Fechas inconsistentes", "Cuarentena si la secuencia es imposible; marcar desfases de registro"),
    "R06": ("Precios inválidos", "Cuarentena: precio <= 0, flete < 0 o pago < 0"),
    "R07": ("Valores atípicos (IQR)", "Conservar y marcar: un atípico no es necesariamente un error"),
    "R08": ("Categorías", "Corregir con catálogo de correspondencias"),
    "R09": ("Integridad referencial", "Cuarentena de huérfanos y, en cascada, de hijos de pedidos en cuarentena"),
    "R10": ("Estados de pedido", "Conservar y marcar contradicciones entre estado y fechas"),
    "R11": ("Códigos postales", "LPAD a 5 dígitos si falta el cero inicial; inválido -> NULL y marcado"),
    "R12": ("Pagos vs ítems", "Conservar y marcar pedidos con diferencia mayor a la tolerancia"),
    "R13": ("Métodos de pago", "Normalizar con CASE; valores desconocidos -> desconocido y marcado"),
    "R14": ("Cadenas vacías", "Corregir: '', espacios y textos tipo N/A pasan a NULL"),
    "R15": ("Validación final", "Todas las verificaciones deben dar 0 violaciones"),
    "M01": ("Completitud", "Porcentaje de celdas sin dato (NULL o vacías)"),
}
METRICAS = {}   # regla -> [(tabla, plantilla SQL con {capa})]; la plantilla devuelve (evaluados, invalidos)

def medir(regla, tabla, etapa, consulta):
    nombre = f"{regla} {REGLAS[regla][0]}"
    politica = REGLAS[regla][1].replace("'", "''")
    run(f"DELETE FROM quality_report WHERE nombre_regla = '{nombre}' "
        f"AND tabla_evaluada = '{tabla}' AND etapa = '{etapa}'")
    run(f"""
        INSERT INTO quality_report
        SELECT '{nombre}', '{tabla}', '{etapa}', evaluados, invalidos,
               round(100.0 * invalidos / nullif(evaluados, 0), 2), '{politica}', current_timestamp
        FROM ({consulta}) AS m(evaluados, invalidos)
    """)

def medir_regla(regla, etapa):
    capa = "raw" if etapa == "antes" else "clean"
    for tabla, plantilla in METRICAS[regla]:
        medir(regla, tabla, etapa, plantilla.replace("{capa}", capa))
    return ver(f"""
        SELECT nombre_regla, tabla_evaluada, etapa, registros_evaluados, registros_invalidos, porcentaje_error
        FROM quality_report
        WHERE nombre_regla LIKE '{regla} %' AND etapa = '{etapa}'
        ORDER BY tabla_evaluada
    """)

## 1. Copia de trabajo con errores controlados

El dataset original está bastante limpio: no tiene duplicados, huérfanos ni precios negativos, así que varias reglas no
encontrarían nada. Por eso se trabaja sobre una copia en `data/trabajo/` con errores conocidos. Los registros a ensuciar
se eligen ordenando por `md5` y no al azar, así cada ejecución mete exactamente los mismos errores. Todo queda anotado
en `errores_inyectados`, y los archivos de `data/original/` no se tocan.

In [6]:
SEMILLA = "miicc409"   # semilla fija -> siempre se eligen los mismos registros

ORDEN_PSEUDOALEATORIO = {
    "pedidos": "order_id",
    "items": "order_id || '-' || order_item_id",
    "clientes": "customer_id",
    "pagos": "order_id || '-' || payment_sequential",
    "productos": "product_id",
}

for tabla, archivo in ARCHIVOS.items():
    ruta = (DIR_ORIGINAL / archivo).as_posix()
    run(f"""
        CREATE OR REPLACE TEMP TABLE w_{tabla} AS
        SELECT *, ROW_NUMBER() OVER (ORDER BY md5({ORDEN_PSEUDOALEATORIO[tabla]} || '{SEMILLA}')) AS _rn
        FROM read_csv('{ruta}', all_varchar = true)
    """)
    n = con.execute(f"SELECT count(*) FROM w_{tabla}").fetchone()[0]
    print(f"w_{tabla:<10} {n:>8,} registros originales")

w_pedidos      99,441 registros originales
w_items       112,650 registros originales
w_clientes     99,441 registros originales
w_pagos       103,886 registros originales
w_productos    32,951 registros originales


In [7]:
# (regla, tabla, descripción, SQL). Los rangos de _rn no se superponen dentro de cada tabla.
INYECCIONES = [
    # ---------------- pedidos ----------------
    ("R01", "pedidos", "Pedido duplicado exacto",
     "INSERT INTO w_pedidos SELECT * FROM w_pedidos WHERE _rn BETWEEN 1 AND 60"),
    ("R01", "pedidos", "Pedido duplicado con el estado en MAYÚSCULAS (solo se detecta por clave)",
     "INSERT INTO w_pedidos SELECT * REPLACE (upper(order_status) AS order_status) FROM w_pedidos WHERE _rn BETWEEN 61 AND 80"),
    ("R03", "pedidos", "Estado con espacios o mayúsculas",
     "UPDATE w_pedidos SET order_status = CASE _rn % 3 WHEN 0 THEN upper(order_status) "
     "WHEN 1 THEN '  ' || order_status || ' ' ELSE upper(left(order_status, 1)) || substr(order_status, 2) END "
     "WHERE _rn BETWEEN 101 AND 400"),
    ("R04", "pedidos", "Fecha de compra en formato dd/mm/aaaa",
     "UPDATE w_pedidos SET order_purchase_timestamp = strftime(CAST(order_purchase_timestamp AS TIMESTAMP), '%d/%m/%Y %H:%M:%S') "
     "WHERE _rn BETWEEN 401 AND 440"),
    ("R04", "pedidos", "Fecha de compra imposible o texto",
     "UPDATE w_pedidos SET order_purchase_timestamp = CASE _rn % 3 WHEN 0 THEN 'fecha_invalida' "
     "WHEN 1 THEN '2017-02-30 10:00:00' ELSE '0000-00-00 00:00:00' END WHERE _rn BETWEEN 441 AND 452"),
    ("R05", "pedidos", "Entrega al cliente anterior a la compra",
     "UPDATE w_pedidos SET order_delivered_customer_date = "
     "strftime(CAST(order_purchase_timestamp AS TIMESTAMP) - INTERVAL 5 DAY, '%Y-%m-%d %H:%M:%S') "
     "WHERE _rn BETWEEN 461 AND 500 AND order_delivered_customer_date IS NOT NULL"),
    ("R05", "pedidos", "Fecha estimada de entrega anterior a la compra",
     "UPDATE w_pedidos SET order_estimated_delivery_date = "
     "strftime(CAST(order_purchase_timestamp AS TIMESTAMP) - INTERVAL 3 DAY, '%Y-%m-%d 00:00:00') "
     "WHERE _rn BETWEEN 501 AND 520"),
    ("R09", "pedidos", "Pedido de un cliente inexistente",
     "UPDATE w_pedidos SET customer_id = md5('cliente-fantasma-' || _rn) WHERE _rn BETWEEN 531 AND 560"),
    ("R14", "pedidos", "Fecha de aprobación vacía, con espacios, 'N/A' o 'null'",
     "UPDATE w_pedidos SET order_approved_at = CASE _rn % 4 WHEN 0 THEN '' WHEN 1 THEN '   ' "
     "WHEN 2 THEN 'N/A' ELSE 'null' END WHERE _rn BETWEEN 561 AND 600"),
    ("R14", "pedidos", "Estado del pedido en blanco",
     "UPDATE w_pedidos SET order_status = '   ' WHERE _rn BETWEEN 601 AND 610"),
    ("R10", "pedidos", "Pedido entregado marcado como cancelado",
     "UPDATE w_pedidos SET order_status = 'canceled' WHERE _rn BETWEEN 611 AND 650 "
     "AND order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL"),
    ("R10", "pedidos", "Pedido 'delivered' sin fecha de entrega",
     "UPDATE w_pedidos SET order_delivered_customer_date = NULL WHERE _rn BETWEEN 651 AND 680 "
     "AND order_status = 'delivered'"),
    # ---------------- items ----------------
    ("R01", "items", "Ítem duplicado (mismo pedido y número de ítem)",
     "INSERT INTO w_items SELECT * FROM w_items WHERE _rn BETWEEN 1 AND 80"),
    ("R06", "items", "Precio igual a cero",
     "UPDATE w_items SET price = CASE WHEN _rn % 2 = 0 THEN '0' ELSE '0.00' END WHERE _rn BETWEEN 101 AND 130"),
    ("R06", "items", "Precio negativo",
     "UPDATE w_items SET price = '-' || price WHERE _rn BETWEEN 131 AND 160"),
    ("R06", "items", "Flete negativo",
     "UPDATE w_items SET freight_value = '-' || freight_value WHERE _rn BETWEEN 161 AND 190 "
     "AND CAST(freight_value AS DOUBLE) > 0"),
    ("R04", "items", "Precio con coma decimal",
     "UPDATE w_items SET price = replace(price, '.', ',') WHERE _rn BETWEEN 201 AND 240"),
    ("R04", "items", "Precio no numérico",
     "UPDATE w_items SET price = CASE _rn % 3 WHEN 0 THEN 'abc' WHEN 1 THEN 'sin precio' ELSE '12.3.4' END "
     "WHERE _rn BETWEEN 241 AND 252"),
    ("R09", "items", "Ítem de un producto inexistente",
     "UPDATE w_items SET product_id = md5('producto-fantasma-' || _rn) WHERE _rn BETWEEN 261 AND 290"),
    ("R09", "items", "Ítem de un pedido inexistente",
     "UPDATE w_items SET order_id = md5('pedido-fantasma-' || _rn) WHERE _rn BETWEEN 291 AND 310"),
    # ---------------- clientes ----------------
    ("R01", "clientes", "Cliente duplicado",
     "INSERT INTO w_clientes SELECT * FROM w_clientes WHERE _rn BETWEEN 1 AND 50"),
    ("R03", "clientes", "Ciudad con mayúsculas o espacios",
     "UPDATE w_clientes SET customer_city = CASE _rn % 3 WHEN 0 THEN upper(customer_city) "
     "WHEN 1 THEN '  ' || customer_city || '  ' ELSE upper(left(customer_city, 1)) || substr(customer_city, 2) END "
     "WHERE _rn BETWEEN 101 AND 400"),
    ("R03", "clientes", "Ciudad con tilde ('são paulo')",
     "UPDATE w_clientes SET customer_city = replace(customer_city, 'sao ', 'são ') "
     "WHERE _rn BETWEEN 401 AND 1000 AND customer_city LIKE 'sao %'"),
    ("R03", "clientes", "Estado (UF) en minúsculas o con espacios",
     "UPDATE w_clientes SET customer_state = CASE _rn % 2 WHEN 0 THEN lower(customer_state) "
     "ELSE ' ' || customer_state || ' ' END WHERE _rn BETWEEN 1001 AND 1200"),
    ("R11", "clientes", "Código postal sin el cero inicial (como lo deja Excel)",
     "UPDATE w_clientes SET customer_zip_code_prefix = ltrim(customer_zip_code_prefix, '0') "
     "WHERE _rn BETWEEN 1201 AND 2400 AND customer_zip_code_prefix LIKE '0%'"),
    ("R11", "clientes", "Código postal inválido",
     "UPDATE w_clientes SET customer_zip_code_prefix = CASE _rn % 3 WHEN 0 THEN 'ABCDE' "
     "WHEN 1 THEN '123456' ELSE '12-34' END WHERE _rn BETWEEN 2401 AND 2415"),
    ("R14", "clientes", "Ciudad vacía, con espacios, 'N/A' o 'NULL'",
     "UPDATE w_clientes SET customer_city = CASE _rn % 4 WHEN 0 THEN '' WHEN 1 THEN '   ' "
     "WHEN 2 THEN 'N/A' ELSE 'NULL' END WHERE _rn BETWEEN 2501 AND 2540"),
    # ---------------- pagos ----------------
    ("R01", "pagos", "Pago duplicado",
     "INSERT INTO w_pagos SELECT * FROM w_pagos WHERE _rn BETWEEN 1 AND 40"),
    ("R13", "pagos", "Método de pago escrito de otra forma",
     "UPDATE w_pagos SET payment_type = CASE payment_type "
     "WHEN 'credit_card' THEN (CASE _rn % 3 WHEN 0 THEN 'Credit Card' WHEN 1 THEN 'CREDIT_CARD' ELSE 'cartao_credito' END) "
     "WHEN 'boleto' THEN (CASE _rn % 2 WHEN 0 THEN ' Boleto ' ELSE 'BOLETO' END) "
     "WHEN 'debit_card' THEN 'debit card' WHEN 'voucher' THEN 'Voucher ' ELSE payment_type END "
     "WHERE _rn BETWEEN 101 AND 500"),
    ("R13", "pagos", "Método de pago desconocido",
     "UPDATE w_pagos SET payment_type = CASE _rn % 3 WHEN 0 THEN 'pix' WHEN 1 THEN 'cripto' ELSE '???' END "
     "WHERE _rn BETWEEN 501 AND 515"),
    ("R04", "pagos", "Monto con coma decimal",
     "UPDATE w_pagos SET payment_value = replace(payment_value, '.', ',') WHERE _rn BETWEEN 521 AND 550"),
    ("R09", "pagos", "Pago de un pedido inexistente",
     "UPDATE w_pagos SET order_id = md5('pedido-fantasma-pago-' || _rn) WHERE _rn BETWEEN 561 AND 580"),
    ("R14", "pagos", "Método de pago vacío o con espacios",
     "UPDATE w_pagos SET payment_type = CASE _rn % 2 WHEN 0 THEN '' ELSE '  ' END WHERE _rn BETWEEN 591 AND 600"),
    ("R02", "pagos", "Monto de pago vacío (campo crítico)",
     "UPDATE w_pagos SET payment_value = '' WHERE _rn BETWEEN 601 AND 610"),
    # ---------------- productos ----------------
    ("R01", "productos", "Producto duplicado",
     "INSERT INTO w_productos SELECT * FROM w_productos WHERE _rn BETWEEN 1 AND 30"),
    ("R03", "productos", "Categoría con mayúsculas o espacios",
     "UPDATE w_productos SET product_category_name = CASE _rn % 2 WHEN 0 THEN upper(product_category_name) "
     "ELSE ' ' || product_category_name || '  ' END WHERE _rn BETWEEN 101 AND 300 AND product_category_name IS NOT NULL"),
    ("R08", "productos", "Categoría con espacios o guiones en lugar de guion bajo",
     "UPDATE w_productos SET product_category_name = CASE _rn % 2 WHEN 0 THEN replace(product_category_name, '_', ' ') "
     "ELSE replace(product_category_name, '_', '-') END "
     "WHERE _rn BETWEEN 301 AND 500 AND contains(product_category_name, '_')"),
    ("R03", "productos", "Categoría con tildes",
     "UPDATE w_productos SET product_category_name = replace(replace(replace(product_category_name, "
     "'informatica', 'informática'), 'moveis', 'móveis'), 'eletrodomesticos', 'eletrodomésticos') "
     "WHERE _rn BETWEEN 501 AND 1500 AND regexp_matches(product_category_name, 'informatica|moveis|eletrodomesticos')"),
    ("R14", "productos", "Categoría vacía, con espacios o 'N/A'",
     "UPDATE w_productos SET product_category_name = CASE _rn % 3 WHEN 0 THEN '' WHEN 1 THEN '  ' ELSE 'N/A' END "
     "WHERE _rn BETWEEN 1601 AND 1630"),
    ("R04", "productos", "Peso no numérico",
     "UPDATE w_productos SET product_weight_g = CASE _rn % 2 WHEN 0 THEN 'sin_dato' ELSE '1,2kg' END "
     "WHERE _rn BETWEEN 1701 AND 1710"),
]

registro = []
for regla, tabla, descripcion, sql in INYECCIONES:
    n = run(sql).fetchone()[0]          # DuckDB devuelve cuántas filas cambió
    registro.append((regla, tabla, descripcion, n))

valores = ",\n        ".join(
    "('{}', '{}', '{}', {})".format(r, t, d.replace("'", "''"), n) for r, t, d, n in registro)
run(f"""
    CREATE OR REPLACE TABLE errores_inyectados AS
    SELECT * FROM (VALUES
        {valores}
    ) AS v(regla, tabla, descripcion, registros_afectados)
""")

# Escribir la copia de trabajo. ORDER BY ALL -> el archivo sale idéntico en cada ejecución.
for tabla, archivo in ARCHIVOS.items():
    destino = (DIR_TRABAJO / archivo).as_posix()
    run(f"COPY (SELECT * EXCLUDE (_rn) FROM w_{tabla} ORDER BY ALL) TO '{destino}' (HEADER, DELIMITER ',')")

ver("SELECT * FROM errores_inyectados ORDER BY regla, tabla")

,regla,tabla,descripcion,registros_afectados
0,R01,clientes,Cliente duplicado,50
1,R01,items,Ítem duplicado (mismo pedido y número de ítem),80
2,R01,pagos,Pago duplicado,40
3,R01,pedidos,Pedido duplicado exacto,60
4,R01,pedidos,Pedido duplicado con el estado en MAYÚSCULAS (solo se detecta por clave),20
5,R01,productos,Producto duplicado,30
6,R02,pagos,Monto de pago vacío (campo crítico),10
7,R03,clientes,Ciudad con mayúsculas o espacios,300
8,R03,clientes,Estado (UF) en minúsculas o con espacios,200
9,R03,clientes,Ciudad con tilde ('são paulo'),138


## 2. Capa RAW

Los CSV de `data/trabajo/` se cargan sin cambiar nada:

- `all_varchar = true` lee todo como texto, para no perder ceros iniciales ni romper la carga por un valor raro.
- `allow_quoted_nulls = false` hace que `""` llegue como cadena vacía y no como `NULL`. Si no, R14 no podría distinguirlos.

También se carga el archivo de traducción de categorías, que es la base del catálogo de R08.

In [8]:
for tabla, archivo in ARCHIVOS.items():
    ruta = (DIR_TRABAJO / archivo).as_posix()
    run(f"""
        CREATE OR REPLACE TABLE {tabla}_raw AS
        SELECT * FROM read_csv('{ruta}', all_varchar = true, allow_quoted_nulls = false)
    """)

ruta_traduccion = (DIR_ORIGINAL / "product_category_name_translation.csv").as_posix()
run(f"""
    CREATE OR REPLACE TABLE categorias_raw AS
    SELECT * FROM read_csv('{ruta_traduccion}', all_varchar = true)
""")

ver(" UNION ALL ".join(f"""
    SELECT '{t}' AS tabla,
           (SELECT count(*) FROM read_csv('{(DIR_ORIGINAL / a).as_posix()}', all_varchar = true)) AS registros_csv_original,
           (SELECT count(*) FROM {t}_raw) AS registros_raw,
           (SELECT count(*) FROM information_schema.columns WHERE table_name = '{t}_raw') AS columnas
""" for t, a in ARCHIVOS.items()))

,tabla,registros_csv_original,registros_raw,columnas
0,pedidos,99441,99521,8
1,items,112650,112730,7
2,clientes,99441,99491,5
3,pagos,103886,103926,5
4,productos,32951,32981,9


In [9]:
# Línea base: todas las columnas llegan como texto (VARCHAR)
ver("""
    SELECT table_name AS tabla, list(column_name ORDER BY ordinal_position) AS columnas,
           list(DISTINCT data_type) AS tipos
    FROM information_schema.columns
    WHERE table_name LIKE '%\\_raw' ESCAPE '\\'
    GROUP BY table_name ORDER BY table_name
""")

,tabla,columnas,tipos
0,categorias_raw,"[product_category_name, product_category_name_english]",[VARCHAR]
1,clientes_raw,"[customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_st...",[VARCHAR]
2,items_raw,"[order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_v...",[VARCHAR]
3,pagos_raw,"[order_id, payment_sequential, payment_type, payment_installments, payment_value]",[VARCHAR]
4,pedidos_raw,"[order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, ord...",[VARCHAR]
5,productos_raw,"[product_id, product_category_name, product_name_lenght, product_description_lenght, p...",[VARCHAR]


## 3. Las 15 reglas de limpieza

Todas siguen el mismo patrón: un diagnóstico sobre RAW, que guarda la métrica "antes" en `quality_report`; la
transformación sobre las tablas `*_stg`, y una comprobación rápida. La métrica "después" se calcula en la sección 5,
ya sobre CLEAN.

### R14 · Cadenas vacías (paso 1)

Un dato que falta puede llegar de varias formas: `NULL`, `''`, `'   '` o un texto como `'N/A'` o `'null'`. Para SQL
no son lo mismo: `IS NULL` no ve la cadena vacía y `COUNT(col)` cuenta `'N/A'` como si fuera un dato.

**Decisión: corregir.** Todo eso pasa a `NULL` con `NULLIF(TRIM(x), '')` (macros `es_vacio` y `limpiar_vacio`).
Va primero para que el resto de las reglas solo tenga que lidiar con `NULL`.

*SQL: `NULLIF`, `TRIM`, `CASE`.*

In [10]:
def perfil_vacios(tabla):
    """Por columna: NULL reales, cadenas vacías, solo espacios y textos que significan 'sin dato'."""
    consultas = [f"""
        SELECT '{tabla}' AS tabla, '{c}' AS columna,
               count(*) FILTER (WHERE "{c}" IS NULL)                         AS nulos,
               count(*) FILTER (WHERE "{c}" = '')                            AS cadena_vacia,
               count(*) FILTER (WHERE "{c}" <> '' AND trim("{c}") = '')      AS solo_espacios,
               count(*) FILTER (WHERE upper(trim("{c}")) IN {TEXTOS_NULOS})  AS texto_nulo
        FROM {tabla}""" for c in columnas(tabla)]
    return ver(" UNION ALL ".join(consultas))

METRICAS["R14"] = [
    (t, f"SELECT count(*), count(*) FILTER (WHERE {cualquiera('es_vacio({c})', columnas(t + '_raw'))}) FROM {t}_{{capa}}")
    for t in TABLAS
]
display(medir_regla("R14", "antes"))

perfil_r14 = pd.concat([perfil_vacios(f"{t}_raw") for t in TABLAS], ignore_index=True)
perfil_r14[perfil_r14[["nulos", "cadena_vacia", "solo_espacios", "texto_nulo"]].sum(axis=1) > 0]

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R14 Cadenas vacías,clientes,antes,99491,40,0.04
1,R14 Cadenas vacías,items,antes,112730,0,0.00
2,R14 Cadenas vacías,pagos,antes,103926,20,0.02
3,R14 Cadenas vacías,pedidos,antes,99521,50,0.05
4,R14 Cadenas vacías,productos,antes,32981,30,0.09


,tabla,columna,nulos,cadena_vacia,solo_espacios,texto_nulo
2,pedidos_raw,order_status,0,0,10,0
4,pedidos_raw,order_approved_at,160,10,10,20
5,pedidos_raw,order_delivered_carrier_date,1784,0,0,0
6,pedidos_raw,order_delivered_customer_date,2997,0,0,0
18,clientes_raw,customer_city,0,10,10,20
22,pagos_raw,payment_type,0,5,5,0
24,pagos_raw,payment_value,0,10,0,0
26,productos_raw,product_category_name,612,10,10,10
27,productos_raw,product_name_lenght,612,0,0,0
28,productos_raw,product_description_lenght,612,0,0,0


In [11]:
# Transformación: se crea la tabla de trabajo *_stg con los vacíos convertidos a NULL
# y dos columnas de control que usarán las demás reglas.
for t in TABLAS:
    seleccion = ",\n               ".join(f'limpiar_vacio("{c}") AS "{c}"' for c in columnas(f"{t}_raw"))
    run(f"""
        CREATE OR REPLACE TABLE {t}_stg AS
        SELECT {seleccion},
               CAST(NULL AS VARCHAR) AS motivo_cuarentena,   -- por qué el registro no pasa a CLEAN
               CAST(NULL AS VARCHAR) AS observaciones        -- problemas conservados y marcados
        FROM {t}_raw
    """)

print("SQL generado para productos:\n")
print(PIPELINE[-1])

# Comprobación: ya no quedan vacíos "disfrazados"
perfil_stg = pd.concat([perfil_vacios(f"{t}_stg") for t in TABLAS], ignore_index=True)
perfil_stg.groupby("tabla")[["cadena_vacia", "solo_espacios", "texto_nulo"]].sum()

SQL generado para productos:

CREATE OR REPLACE TABLE productos_stg AS
SELECT limpiar_vacio("product_id") AS "product_id",
       limpiar_vacio("product_category_name") AS "product_category_name",
       limpiar_vacio("product_name_lenght") AS "product_name_lenght",
       limpiar_vacio("product_description_lenght") AS "product_description_lenght",
       limpiar_vacio("product_photos_qty") AS "product_photos_qty",
       limpiar_vacio("product_weight_g") AS "product_weight_g",
       limpiar_vacio("product_length_cm") AS "product_length_cm",
       limpiar_vacio("product_height_cm") AS "product_height_cm",
       limpiar_vacio("product_width_cm") AS "product_width_cm",
       CAST(NULL AS VARCHAR) AS motivo_cuarentena,   -- por qué el registro no pasa a CLEAN
       CAST(NULL AS VARCHAR) AS observaciones        -- problemas conservados y marcados
FROM productos_raw


,cadena_vacia,solo_espacios,texto_nulo
tabla,,,
clientes_stg,0,0,0
items_stg,0,0,0
pagos_stg,0,0,0
pedidos_stg,0,0,0
productos_stg,0,0,0


### R03 · Estandarización de texto (paso 2)

Un mismo valor aparece escrito de varias maneras: `' DELIVERED'`, `'Delivered'`, `'são paulo'` y `'sao paulo'`,
`' sp'`. Para un `GROUP BY` o un `JOIN` son valores distintos.

**Decisión: corregir.** Estados, ciudades, categorías y métodos de pago quedan en minúsculas, sin espacios de más y sin
tildes, que es como vienen en el dataset original (macro `normalizar_texto`). La UF va en mayúsculas porque es una
sigla. Es un arreglo seguro: el valor no cambia de significado.

*SQL: `TRIM`, `LOWER`, `UPPER`, `strip_accents`, `regexp_replace`.*

In [12]:
METRICAS["R03"] = [
    ("pedidos", "SELECT count(*), count(*) FILTER (WHERE order_status <> normalizar_texto(order_status) "
                "AND NOT es_vacio(order_status)) FROM pedidos_{capa}"),
    ("clientes", "SELECT count(*), count(*) FILTER (WHERE (customer_city <> normalizar_texto(customer_city) "
                 "AND NOT es_vacio(customer_city)) OR customer_state <> upper(trim(customer_state))) FROM clientes_{capa}"),
    ("productos", "SELECT count(*), count(*) FILTER (WHERE product_category_name <> normalizar_texto(product_category_name) "
                  "AND NOT es_vacio(product_category_name)) FROM productos_{capa}"),
    ("pagos", "SELECT count(*), count(*) FILTER (WHERE payment_type <> normalizar_texto(payment_type) "
              "AND NOT es_vacio(payment_type)) FROM pagos_{capa}"),
]
display(medir_regla("R03", "antes"))

DISTINTOS_TEXTO = """
    SELECT 'pedidos.order_status' AS campo, count(DISTINCT order_status) AS valores_distintos FROM pedidos_stg
    UNION ALL SELECT 'clientes.customer_city', count(DISTINCT customer_city) FROM clientes_stg
    UNION ALL SELECT 'clientes.customer_state', count(DISTINCT customer_state) FROM clientes_stg
    UNION ALL SELECT 'productos.product_category_name', count(DISTINCT product_category_name) FROM productos_stg
    UNION ALL SELECT 'pagos.payment_type', count(DISTINCT payment_type) FROM pagos_stg
"""
distintos_antes = ver(DISTINTOS_TEXTO)
ver("SELECT order_status, count(*) AS pedidos FROM pedidos_stg GROUP BY ALL ORDER BY pedidos DESC")

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R03 Estandarización de texto,clientes,antes,99491,638,0.64
1,R03 Estandarización de texto,pagos,antes,103926,286,0.28
2,R03 Estandarización de texto,pedidos,antes,99521,320,0.32
3,R03 Estandarización de texto,productos,antes,32981,360,1.09


,order_status,pedidos
0,delivered,96193
1,shipped,1105
2,canceled,664
3,unavailable,610
4,invoiced,311
5,processing,301
6,DELIVERED,117
7,delivered,99
8,Delivered,98
9,NaN,10


In [13]:
run("UPDATE pedidos_stg   SET order_status = normalizar_texto(order_status)")
run("UPDATE clientes_stg  SET customer_city = normalizar_texto(customer_city), customer_state = upper(trim(customer_state))")
run("UPDATE productos_stg SET product_category_name = normalizar_texto(product_category_name)")
run("UPDATE pagos_stg     SET payment_type = normalizar_texto(payment_type)")

# Comprobación: menos valores distintos para lo mismo
comparacion = distintos_antes.merge(ver(DISTINTOS_TEXTO), on="campo", suffixes=("_antes", "_despues"))
display(comparacion)
ver("SELECT order_status, count(*) AS pedidos FROM pedidos_stg GROUP BY ALL ORDER BY pedidos DESC")

,campo,valores_distintos_antes,valores_distintos_despues
0,pedidos.order_status,16,8
1,clientes.customer_city,4334,4119
2,clientes.customer_state,60,27
3,productos.product_category_name,181,114
4,pagos.payment_type,15,11


,order_status,pedidos
0,delivered,96507
1,shipped,1108
2,canceled,664
3,unavailable,610
4,invoiced,314
5,processing,301
6,NaN,10
7,created,5
8,approved,2


### R04 · Tipos de datos (paso 3)

Todo llegó como texto, y no todo se convierte limpio: hay fechas como `02/10/2017 10:56:33`, fechas que no existen
(`2017-02-30`, `0000-00-00`), texto en columnas de fecha, precios con coma (`58,90`) y cosas como `abc` o `1,2kg`.

**Decisión.** Se convierte con `TRY_CAST` y `TRY_STRPTIME`, que devuelven `NULL` en vez de fallar. Se aceptan los dos
formatos de fecha (`aaaa-mm-dd` y el `dd/mm/aaaa` que se usa en Brasil) y la coma decimal, así que esos valores se
recuperan. Con lo que aun así no se convierte, depende del campo:

- si es crítico (fecha de compra, precio, flete, monto, claves), el registro va a cuarentena: no hay forma de saber el valor real;
- si no lo es, como el peso de un producto, queda en `NULL`.

El código postal se deja como texto a propósito: como número perdería los ceros iniciales (ver R11).

*SQL: `TRY_CAST`, `TRY_STRPTIME`, `ALTER COLUMN ... TYPE ... USING`.*

In [14]:
diagnostico = []
for t, cols in CONVERSIONES.items():
    for c, tipo in cols.items():
        conv = conversion(c, tipo)
        diagnostico.append(f"""
            SELECT '{t}' AS tabla, '{c}' AS columna, '{tipo}' AS tipo_destino,
                   count("{c}") AS con_valor,
                   count(*) FILTER (WHERE "{c}" IS NOT NULL AND TRY_CAST("{c}" AS {tipo}) IS NULL
                                    AND {conv} IS NOT NULL) AS recuperables_por_formato,
                   count(*) FILTER (WHERE "{c}" IS NOT NULL AND {conv} IS NULL) AS no_convertibles,
                   list(DISTINCT "{c}") FILTER (WHERE "{c}" IS NOT NULL AND {conv} IS NULL) AS ejemplos_no_convertibles
            FROM {t}_stg""")
r04 = ver(" UNION ALL ".join(diagnostico))

METRICAS["R04"] = []
for t, cols in CONVERSIONES.items():
    invalido = " OR ".join(f'("{c}" IS NOT NULL AND NOT es_vacio("{c}") AND {conversion(c, tipo)} IS NULL)'
                           for c, tipo in cols.items())
    METRICAS["R04"].append((t, f"SELECT count(*), count(*) FILTER (WHERE {invalido}) FROM {t}_{{capa}}"))
display(medir_regla("R04", "antes"))

r04[(r04.recuperables_por_formato > 0) | (r04.no_convertibles > 0)]

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R04 Tipos de datos,items,antes,112730,12,0.01
1,R04 Tipos de datos,pagos,antes,103926,0,0.00
2,R04 Tipos de datos,pedidos,antes,99521,12,0.01
3,R04 Tipos de datos,productos,antes,32981,10,0.03


,tabla,columna,tipo_destino,con_valor,recuperables_por_formato,no_convertibles,ejemplos_no_convertibles
0,pedidos,order_purchase_timestamp,TIMESTAMP,99521,40,12,"[0000-00-00 00:00:00, 2017-02-30 10:00:00, fecha_invalida]"
7,items,price,"DECIMAL(10,2)",112730,40,12,"[12.3.4, abc, sin precio]"
11,pagos,payment_value,"DECIMAL(10,2)",103916,30,0,<NA>
15,productos,product_weight_g,INTEGER,32979,0,10,"[sin_dato, 1,2kg]"


In [15]:
for t, cols in CONVERSIONES.items():
    for c, tipo in cols.items():
        conv = conversion(c, tipo)
        if c in CRITICAS[t]:
            marcar_cuarentena(f"{t}_stg", f"R04 {c} no convertible a {tipo}", f'"{c}" IS NOT NULL AND {conv} IS NULL')
        run(f'ALTER TABLE {t}_stg ALTER COLUMN "{c}" TYPE {tipo} USING {conv}')

# Comprobación: tipos finales de las columnas convertidas
ver("""
    SELECT table_name AS tabla, column_name AS columna, data_type AS tipo
    FROM information_schema.columns
    WHERE table_name LIKE '%\\_stg' ESCAPE '\\' AND data_type <> 'VARCHAR'
    ORDER BY table_name, ordinal_position
""")

  cuarentena  pedidos_stg        12  R04 order_purchase_timestamp no convertible a TIMESTAMP
  cuarentena  items_stg           0  R04 order_item_id no convertible a INTEGER
  cuarentena  items_stg          12  R04 price no convertible a DECIMAL(10,2)
  cuarentena  items_stg           0  R04 freight_value no convertible a DECIMAL(10,2)
  cuarentena  pagos_stg           0  R04 payment_sequential no convertible a INTEGER
  cuarentena  pagos_stg           0  R04 payment_value no convertible a DECIMAL(10,2)


,tabla,columna,tipo
0,items_stg,order_item_id,INTEGER
1,items_stg,shipping_limit_date,TIMESTAMP
2,items_stg,price,"DECIMAL(10,2)"
3,items_stg,freight_value,"DECIMAL(10,2)"
4,pagos_stg,payment_sequential,INTEGER
5,pagos_stg,payment_installments,INTEGER
6,pagos_stg,payment_value,"DECIMAL(10,2)"
7,pedidos_stg,order_purchase_timestamp,TIMESTAMP
8,pedidos_stg,order_approved_at,TIMESTAMP
9,pedidos_stg,order_delivered_carrier_date,TIMESTAMP


### R01 · Duplicados (paso 4)

Los duplicados inflan conteos y montos. El cuidado está en `items`: ahí un `order_id` se repite con razón, porque un
pedido puede tener varios productos. Por eso cada tabla se deduplica por su clave de negocio:

| Tabla | Clave de negocio |
|---|---|
| pedidos | `order_id` |
| items | `order_id` + `order_item_id` |
| clientes | `customer_id` (`customer_unique_id` sí se repite: es la misma persona en otro pedido) |
| pagos | `order_id` + `payment_sequential` |
| productos | `product_id` |

**Decisión: corregir.** Se deja una fila por clave con `ROW_NUMBER() OVER (PARTITION BY ...)`: primero la que no
tiene problemas, luego la más completa y, si siguen empatadas, un desempate fijo con `md5`. Como R03 ya normalizó el
texto, también caen duplicados como `delivered` / `DELIVERED`, que un `SELECT DISTINCT *` no habría encontrado.

*SQL: `ROW_NUMBER`, `PARTITION BY`, `QUALIFY`.*

In [16]:
def resumen_duplicados(capa):
    consultas = []
    for t, k in CLAVES_NEGOCIO.items():
        clave = ", ".join(k)
        consultas.append(f"""
            SELECT '{t}' AS tabla, '{clave}' AS clave_negocio,
                   (SELECT count(*) FROM {t}_{capa}) AS registros,
                   (SELECT count(*) FROM (SELECT DISTINCT {clave} FROM {t}_{capa})) AS claves_unicas,
                   (SELECT count(*) FROM (SELECT DISTINCT * FROM {t}_{capa})) AS filas_distintas""")
    df = ver(" UNION ALL ".join(consultas))
    df["duplicados_por_clave"] = df.registros - df.claves_unicas
    df["duplicados_exactos"] = df.registros - df.filas_distintas
    return df.drop(columns="filas_distintas")

METRICAS["R01"] = [
    (t, f"SELECT count(*), count(*) - (SELECT count(*) FROM (SELECT DISTINCT {', '.join(k)} FROM {t}_{{capa}})) "
        f"FROM {t}_{{capa}}")
    for t, k in CLAVES_NEGOCIO.items()
]
display(medir_regla("R01", "antes"))
display(resumen_duplicados("raw"))

# Por qué NO se puede deduplicar items solo por order_id:
display(ver("""
    SELECT count(*) AS items,
           count(DISTINCT order_id) AS pedidos_distintos,
           count(*) - count(DISTINCT order_id) AS falsos_duplicados_si_se_usara_solo_order_id
    FROM items_raw
"""))

# Duplicados que solo difieren en formato (en RAW no son filas idénticas)
ver("""
    SELECT order_id, order_status, order_purchase_timestamp,
           ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_status) AS n_copia
    FROM pedidos_raw
    QUALIFY count(*) OVER (PARTITION BY order_id) > 1
        AND count(DISTINCT order_status) OVER (PARTITION BY order_id) > 1
    ORDER BY order_id, n_copia
    LIMIT 6
""")

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R01 Duplicados,clientes,antes,99491,50,0.05
1,R01 Duplicados,items,antes,112730,80,0.07
2,R01 Duplicados,pagos,antes,103926,40,0.04
3,R01 Duplicados,pedidos,antes,99521,80,0.08
4,R01 Duplicados,productos,antes,32981,30,0.09


,tabla,clave_negocio,registros,claves_unicas,duplicados_por_clave,duplicados_exactos
0,pedidos,order_id,99521,99441,80,60
1,items,"order_id, order_item_id",112730,112650,80,80
2,clientes,customer_id,99491,99441,50,50
3,pagos,"order_id, payment_sequential",103926,103886,40,40
4,productos,product_id,32981,32951,30,30


,items,pedidos_distintos,falsos_duplicados_si_se_usara_solo_order_id
0,112730,98670,14060


,order_id,order_status,order_purchase_timestamp,n_copia
0,024a74550cdbd36c0fe3955fc9e2bcf0,DELIVERED,2018-08-09 08:10:23,1
1,024a74550cdbd36c0fe3955fc9e2bcf0,delivered,2018-08-09 08:10:23,2
2,189f4fd5230d4cb9093778f1fe5545dd,DELIVERED,2018-07-14 20:30:45,1
3,189f4fd5230d4cb9093778f1fe5545dd,delivered,2018-07-14 20:30:45,2
4,23f592266a54ffc371a5c3eee760aaae,DELIVERED,2017-11-29 18:59:32,1
5,23f592266a54ffc371a5c3eee760aaae,delivered,2017-11-29 18:59:32,2


In [17]:
for t, k in CLAVES_NEGOCIO.items():
    clave = ", ".join(k)
    nulos = " + ".join(f'("{c}" IS NULL)::INT' for c in columnas(f"{t}_stg")
                       if c not in ("motivo_cuarentena", "observaciones"))
    run(f"""
        CREATE OR REPLACE TABLE {t}_stg AS
        SELECT * EXCLUDE (n_copia)
        FROM (
            SELECT s.*,
                   ROW_NUMBER() OVER (
                       PARTITION BY {clave}
                       ORDER BY (s.motivo_cuarentena IS NOT NULL),  -- 1) la copia sin problemas
                                {nulos},  -- 2) la más completa
                                md5(CAST(s AS VARCHAR))             -- 3) desempate determinista
                   ) AS n_copia
            FROM {t}_stg AS s
        )
        WHERE n_copia = 1
    """)

# Comprobación: 0 duplicados por clave en staging
resumen_duplicados("stg")

,tabla,clave_negocio,registros,claves_unicas,duplicados_por_clave,duplicados_exactos
0,pedidos,order_id,99441,99441,0,0
1,items,"order_id, order_item_id",112650,112650,0,0
2,clientes,customer_id,99441,99441,0,0
3,pagos,"order_id, payment_sequential",103886,103886,0,0
4,productos,product_id,32951,32951,0,0


### R11 · Códigos postales (paso 5)

`customer_zip_code_prefix` son los primeros 5 dígitos del CEP. Si en algún momento se trató como número (o pasó por
Excel), perdió el cero inicial: `01037` queda como `1037`. También hay valores sin sentido, como `ABCDE` o `123456`.

**Decisión.** Los CEP de Brasil van de `01` a `99`, así que como mucho se pierde un cero. Un valor de 4 dígitos se
arregla sin riesgo con `LPAD(x, 5, '0')`. Cualquier otro formato se considera inválido: pasa a `NULL` y el cliente se
marca con `cp_invalido`. No va a cuarentena, porque sus pedidos siguen siendo válidos. La columna se queda como `VARCHAR`.

*SQL: `REGEXP_MATCHES`, `LPAD`.*

In [18]:
METRICAS["R11"] = [
    ("clientes", "SELECT count(*), count(*) FILTER (WHERE customer_zip_code_prefix IS NULL "
                 "OR NOT regexp_matches(customer_zip_code_prefix, '^[0-9]{5}$')) FROM clientes_{capa}"),
]
display(medir_regla("R11", "antes"))

ver("""
    SELECT CASE
               WHEN customer_zip_code_prefix IS NULL THEN 'nulo'
               WHEN regexp_matches(customer_zip_code_prefix, '^[0-9]{5}$') THEN 'válido: 5 dígitos'
               WHEN regexp_matches(customer_zip_code_prefix, '^[0-9]{4}$') THEN 'perdió el cero inicial: 4 dígitos'
               ELSE 'inválido'
           END AS diagnostico,
           count(*) AS clientes,
           list(DISTINCT customer_zip_code_prefix)[1:5] AS ejemplos
    FROM clientes_stg
    GROUP BY diagnostico ORDER BY clientes DESC
""")

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R11 Códigos postales,clientes,antes,99491,304,0.31


,diagnostico,clientes,ejemplos
0,válido: 5 dígitos,99137,"[08223, 09351, 13510, 22250, 12704]"
1,perdió el cero inicial: 4 dígitos,289,"[9530, 1452, 6542, 8090, 5661]"
2,inválido,15,"[12-34, ABCDE, 123456]"


In [19]:
marcar_observacion("clientes_stg", "cp_corregido", "R11 código postal completado con LPAD",
                   "regexp_matches(trim(customer_zip_code_prefix), '^[0-9]{4}$')")
marcar_observacion("clientes_stg", "cp_invalido", "R11 código postal inválido",
                   "customer_zip_code_prefix IS NOT NULL "
                   "AND NOT regexp_matches(trim(customer_zip_code_prefix), '^[0-9]{4,5}$')")
run("""
    UPDATE clientes_stg
    SET customer_zip_code_prefix = CASE
            WHEN regexp_matches(trim(customer_zip_code_prefix), '^[0-9]{4,5}$')
            THEN lpad(trim(customer_zip_code_prefix), 5, '0')
        END   -- inválido -> NULL
""")

ver("""
    SELECT length(customer_zip_code_prefix) AS digitos, count(*) AS clientes,
           count(*) FILTER (WHERE cp_corregido) AS corregidos_con_lpad,
           count(*) FILTER (WHERE cp_invalido) AS invalidos
    FROM clientes_stg GROUP BY ALL ORDER BY ALL
""")

  marcado     clientes_stg      289  R11 código postal completado con LPAD
  marcado     clientes_stg       15  R11 código postal inválido


,digitos,clientes,corregidos_con_lpad,invalidos
0,5,99426,289,0
1,<NA>,15,0,15


### R02 · Valores nulos (paso 6)

Aquí se mide cuántos `NULL` hay por columna y se decide qué hacer con cada una. Se mide en staging, después de R14 y
R04, porque esas reglas son las que convirtieron blancos y valores inválidos en `NULL`.

| Columna | Decisión | Por qué |
|---|---|---|
| Campos críticos (claves, fecha de compra, precio, flete, monto) | cuarentena | sin ellos el registro no sirve y no se pueden deducir |
| Fechas de aprobación, envío y entrega | dejar `NULL` | el `NULL` dice algo: el pedido todavía no llegó a esa etapa |
| `customer_city` | imputar desde el código postal | solo si ese prefijo tiene una única ciudad; si no, queda `NULL` |
| `product_category_name` | imputar `'sin_categoria'` y marcar | se puede agrupar sin inventar una categoría |
| Descripción, fotos, medidas y peso | dejar `NULL` | inventar medidas sesgaría cualquier cálculo de flete |

*SQL: `IS NULL`, `COALESCE`, `UPDATE ... FROM`.*

In [20]:
def perfil_nulos(tabla):
    cols = [c for c in columnas(tabla) if c not in ("motivo_cuarentena", "observaciones")]
    return ver(" UNION ALL ".join(
        f"""SELECT '{c}' AS columna, count(*) AS registros, count(*) FILTER (WHERE "{c}" IS NULL) AS nulos FROM {tabla}"""
        for c in cols))

POLITICA_NULOS = {
    "order_approved_at": "Conservar NULL: el pedido no fue aprobado",
    "order_delivered_carrier_date": "Conservar NULL: el pedido no llegó a esa etapa",
    "order_delivered_customer_date": "Conservar NULL: el pedido no llegó a esa etapa",
    "order_status": "R10: pasa a 'desconocido'",
    "customer_city": "Imputar desde el código postal si es inequívoco",
    "customer_zip_code_prefix": "Conservar NULL (inválido según R11)",
    "product_category_name": "Imputar 'sin_categoria' y marcar",
    "payment_type": "R13: pasa a 'desconocido'",
}

partes = []
for t in TABLAS:
    df = (perfil_nulos(f"{t}_raw").rename(columns={"registros": "registros_raw", "nulos": "nulos_raw"})
          .merge(perfil_nulos(f"{t}_stg").rename(columns={"registros": "registros_stg", "nulos": "nulos_stg"}),
                 on="columna"))
    df.insert(0, "tabla", t)
    partes.append(df)
r02 = pd.concat(partes, ignore_index=True)
r02["pct_nulos_stg"] = (100 * r02.nulos_stg / r02.registros_stg).round(3)
r02["critica"] = [c in CRITICAS[t] for t, c in zip(r02.tabla, r02.columna)]
r02["politica"] = ["Cuarentena (campo crítico)" if crit else POLITICA_NULOS.get(c, "Conservar NULL: no se inventan datos")
                   for c, crit in zip(r02.columna, r02.critica)]

METRICAS["R02"] = [
    (t, f"SELECT count(*), count(*) FILTER (WHERE {cualquiera('({c} IS NULL OR es_vacio({c}))', CRITICAS[t])}) "
        f"FROM {t}_{{capa}}")
    for t in TABLAS
]
display(medir_regla("R02", "antes"))

r02[r02.nulos_stg > 0]

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R02 Valores nulos,clientes,antes,99491,0,0.00
1,R02 Valores nulos,items,antes,112730,0,0.00
2,R02 Valores nulos,pagos,antes,103926,10,0.01
3,R02 Valores nulos,pedidos,antes,99521,0,0.00
4,R02 Valores nulos,productos,antes,32981,0,0.00


,tabla,columna,registros_raw,nulos_raw,registros_stg,nulos_stg,pct_nulos_stg,critica,politica
2,pedidos,order_status,99521,0,99441,10,0.010,False,R10: pasa a 'desconocido'
3,pedidos,order_purchase_timestamp,99521,0,99441,12,0.012,True,Cuarentena (campo crítico)
4,pedidos,order_approved_at,99521,160,99441,200,0.201,False,Conservar NULL: el pedido no fue aprobado
5,pedidos,order_delivered_carrier_date,99521,1784,99441,1783,1.793,False,Conservar NULL: el pedido no llegó a esa etapa
6,pedidos,order_delivered_customer_date,99521,2997,99441,2995,3.012,False,Conservar NULL: el pedido no llegó a esa etapa
13,items,price,112730,0,112650,12,0.011,True,Cuarentena (campo crítico)
17,clientes,customer_zip_code_prefix,99491,0,99441,15,0.015,False,Conservar NULL (inválido según R11)
18,clientes,customer_city,99491,0,99441,40,0.040,False,Imputar desde el código postal si es inequívoco
22,pagos,payment_type,103926,0,103886,10,0.010,False,R13: pasa a 'desconocido'
24,pagos,payment_value,103926,0,103886,10,0.010,True,Cuarentena (campo crítico)


In [21]:
# 1) Campos críticos nulos -> cuarentena (si R04 ya explicó el motivo, no se repite)
for t, cols in CRITICAS.items():
    cond = " OR ".join(f'"{c}" IS NULL' for c in cols)
    marcar_cuarentena(f"{t}_stg", "R02 campo crítico nulo", f"({cond}) AND motivo_cuarentena IS NULL")

# 2) Ciudad: imputar desde el código postal cuando ese prefijo tiene una sola ciudad
run("""
    CREATE OR REPLACE TEMP TABLE ciudad_por_cp AS
    SELECT customer_zip_code_prefix, any_value(customer_city) AS ciudad
    FROM clientes_stg
    WHERE customer_city IS NOT NULL AND customer_zip_code_prefix IS NOT NULL
    GROUP BY customer_zip_code_prefix
    HAVING count(DISTINCT customer_city) = 1
""")
marcar_observacion("clientes_stg", "ciudad_imputada", "R02 ciudad imputada desde el código postal",
                   "customer_city IS NULL AND customer_zip_code_prefix IN (SELECT customer_zip_code_prefix FROM ciudad_por_cp)")
run("""
    UPDATE clientes_stg AS c
    SET customer_city = z.ciudad
    FROM ciudad_por_cp AS z
    WHERE c.customer_city IS NULL AND c.customer_zip_code_prefix = z.customer_zip_code_prefix
""")

# 3) Categoría: imputar 'sin_categoria'
marcar_observacion("productos_stg", "categoria_imputada", "R02 categoría imputada como sin_categoria",
                   "product_category_name IS NULL")
run("UPDATE productos_stg SET product_category_name = COALESCE(product_category_name, 'sin_categoria')")

ver("""
    SELECT 'clientes sin ciudad' AS caso, count(*) AS registros FROM clientes_stg WHERE customer_city IS NULL
    UNION ALL SELECT 'productos sin categoría', count(*) FROM productos_stg WHERE product_category_name IS NULL
    UNION ALL SELECT 'pedidos en cuarentena', count(*) FROM pedidos_stg WHERE motivo_cuarentena IS NOT NULL
    UNION ALL SELECT 'ítems en cuarentena', count(*) FROM items_stg WHERE motivo_cuarentena IS NOT NULL
    UNION ALL SELECT 'pagos en cuarentena', count(*) FROM pagos_stg WHERE motivo_cuarentena IS NOT NULL
""")

  cuarentena  pedidos_stg         0  R02 campo crítico nulo
  cuarentena  items_stg           0  R02 campo crítico nulo
  cuarentena  clientes_stg        0  R02 campo crítico nulo
  cuarentena  pagos_stg          10  R02 campo crítico nulo
  cuarentena  productos_stg       0  R02 campo crítico nulo
  marcado     clientes_stg       40  R02 ciudad imputada desde el código postal
  marcado     productos_stg     640  R02 categoría imputada como sin_categoria


,caso,registros
0,clientes sin ciudad,0
1,productos sin categoría,0
2,pedidos en cuarentena,12
3,ítems en cuarentena,12
4,pagos en cuarentena,10


### R08 · Categorías (paso 7)

La misma categoría aparece con nombres distintos. Algunos casos son de formato (`cama mesa banho` o `cama-mesa-banho`
en vez de `cama_mesa_banho`), pero otros vienen del dataset original: `casa_conforto_2` es lo mismo que
`casa_conforto`, y `eletrodomesticos_2` lo mismo que `eletrodomesticos`. Además, `pc_gamer` y
`portateis_cozinha_e_preparadores_de_alimentos` no están en el archivo de traducción.

**Decisión: corregir con un catálogo.** La tabla `catalogo_categorias` relaciona cada nombre original con su nombre
estándar y su traducción al inglés. Parte del archivo oficial de traducción, más las equivalencias y traducciones que
faltaban. Así, agregar una equivalencia nueva es agregar una fila, sin tocar el código.

*SQL: `REPLACE`, `CASE`, `COALESCE`, `UPDATE ... FROM`.*

In [22]:
run("""
CREATE OR REPLACE TABLE catalogo_categorias AS
WITH traduccion AS (
    -- Archivo de traducción oficial del dataset
    SELECT normalizar_texto(product_category_name) AS categoria,
           normalizar_texto(product_category_name_english) AS categoria_en
    FROM categorias_raw
    UNION ALL
    -- Categorías presentes en productos pero ausentes del archivo de traducción
    SELECT * FROM (VALUES
        ('pc_gamer', 'pc_gamer'),
        ('portateis_cozinha_e_preparadores_de_alimentos', 'portable_kitchen_and_food_preparers'),
        ('sin_categoria', 'uncategorized')
    ) AS v(categoria, categoria_en)
),
equivalencias AS (
    -- El sufijo _2 es la misma categoría duplicada en el catálogo de Olist
    SELECT * FROM (VALUES
        ('casa_conforto_2', 'casa_conforto'),
        ('eletrodomesticos_2', 'eletrodomesticos')
    ) AS v(categoria, categoria_estandar)
)
SELECT t.categoria                                 AS categoria_original,
       coalesce(e.categoria_estandar, t.categoria) AS categoria_estandar,
       coalesce(t2.categoria_en, t.categoria_en)   AS categoria_en
FROM traduccion AS t
LEFT JOIN equivalencias AS e  ON e.categoria = t.categoria
LEFT JOIN traduccion    AS t2 ON t2.categoria = e.categoria_estandar
""")

METRICAS["R08"] = [
    ("productos", "SELECT count(*), count(*) FILTER (WHERE NOT es_vacio(product_category_name) "
                  "AND product_category_name NOT IN (SELECT categoria_estandar FROM catalogo_categorias)) "
                  "FROM productos_{capa}"),
]
display(medir_regla("R08", "antes"))

# Categorías de productos que no son un nombre estándar del catálogo
ver("""
    SELECT p.product_category_name AS categoria_en_productos, count(*) AS productos,
           c.categoria_estandar AS se_unifica_como
    FROM productos_stg AS p
    LEFT JOIN catalogo_categorias AS c ON c.categoria_original = p.product_category_name
    WHERE c.categoria_original IS NULL OR c.categoria_original <> c.categoria_estandar
    GROUP BY ALL ORDER BY productos DESC
    LIMIT 15
""")

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R08 Categorías,productos,antes,32981,595,1.8


,categoria_en_productos,productos,se_unifica_como
0,eletrodomesticos_2,90,eletrodomesticos
1,cama mesa banho,12,NaN
2,cama-mesa-banho,11,NaN
3,utilidades-domesticas,10,NaN
4,esporte lazer,8,NaN
5,moveis decoracao,8,NaN
6,informatica acessorios,7,NaN
7,beleza saude,6,NaN
8,beleza-saude,6,NaN
9,relogios presentes,6,NaN


In [23]:
categorias_antes = ver("SELECT count(DISTINCT product_category_name) AS n FROM productos_stg").n[0]

# 1) Separadores: espacios y guiones -> guion bajo
run("UPDATE productos_stg SET product_category_name = replace(replace(product_category_name, ' ', '_'), '-', '_')")

# 2) Aplicar el catálogo: nombre estándar + traducción al inglés
run("ALTER TABLE productos_stg ADD COLUMN IF NOT EXISTS product_category_name_english VARCHAR")
run("""
    UPDATE productos_stg AS p
    SET product_category_name = c.categoria_estandar,
        product_category_name_english = c.categoria_en
    FROM catalogo_categorias AS c
    WHERE p.product_category_name = c.categoria_original
""")
marcar_observacion("productos_stg", "categoria_no_catalogada", "R08 categoría fuera del catálogo",
                   "product_category_name_english IS NULL")

categorias_despues = ver("SELECT count(DISTINCT product_category_name) AS n FROM productos_stg").n[0]
print(f"Categorías distintas: {categorias_antes} -> {categorias_despues}")
ver("SELECT * FROM catalogo_categorias WHERE categoria_original <> categoria_estandar OR categoria_original IN ('pc_gamer', 'sin_categoria')")

  marcado     productos_stg       0  R08 categoría fuera del catálogo
Categorías distintas: 115 -> 72


,categoria_original,categoria_estandar,categoria_en
0,eletrodomesticos_2,eletrodomesticos,home_appliances
1,casa_conforto_2,casa_conforto,home_confort
2,pc_gamer,pc_gamer,pc_gamer
3,sin_categoria,sin_categoria,uncategorized


### R13 · Métodos de pago (paso 8)

Los métodos válidos son `credit_card`, `boleto`, `voucher` y `debit_card`. Llegan variantes (`Credit Card`,
`CREDIT_CARD`, `cartao_credito`, `debit card`) y valores que no se reconocen (`pix`, `cripto`, `???`, vacíos).
El original además trae 3 pagos con `not_defined`.

**Decisión.** Las variantes conocidas se corrigen con un `CASE`. Lo que no se reconoce pasa a `'desconocido'` y se
marca con `metodo_desconocido`, pero el pago se queda: el monto es real y sacarlo descuadraría el pedido en R12.

*SQL: `DISTINCT`, `CASE`.*

In [24]:
METRICAS["R13"] = [
    ("pagos", f"SELECT count(*), count(*) FILTER (WHERE coalesce(payment_type, '') NOT IN {lista_sql(METODOS_VALIDOS)}) "
              "FROM pagos_{capa}"),
]
display(medir_regla("R13", "antes"))

NORMALIZAR_METODO = """
    CASE replace(payment_type, ' ', '_')
        WHEN 'credit_card'    THEN 'credit_card'
        WHEN 'creditcard'     THEN 'credit_card'
        WHEN 'cartao_credito' THEN 'credit_card'
        WHEN 'debit_card'     THEN 'debit_card'
        WHEN 'cartao_debito'  THEN 'debit_card'
        WHEN 'boleto'         THEN 'boleto'
        WHEN 'voucher'        THEN 'voucher'
        ELSE 'desconocido'    -- not_defined, valores nuevos o NULL
    END"""

ver(f"""
    SELECT DISTINCT payment_type AS valor_actual, {NORMALIZAR_METODO} AS metodo_normalizado,
           count(*) OVER (PARTITION BY payment_type) AS pagos
    FROM pagos_stg
    ORDER BY metodo_normalizado, pagos DESC
""")

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R13 Métodos de pago,pagos,antes,103926,428,0.41


,valor_actual,metodo_normalizado,pagos
0,boleto,boleto,19781
1,credit_card,credit_card,76570
2,cartao_credito,credit_card,106
3,credit card,credit_card,101
4,debit_card,debit_card,1520
5,debit card,debit_card,8
6,NaN,desconocido,10
7,cripto,desconocido,5
8,pix,desconocido,5
9,???,desconocido,5


In [25]:
marcar_observacion("pagos_stg", "metodo_desconocido", "R13 método de pago desconocido",
                   f"({NORMALIZAR_METODO}) = 'desconocido'")
run(f"UPDATE pagos_stg SET payment_type = {NORMALIZAR_METODO}")

ver("SELECT payment_type, count(*) AS pagos FROM pagos_stg GROUP BY ALL ORDER BY pagos DESC")

  marcado     pagos_stg          28  R13 método de pago desconocido


,payment_type,pagos
0,credit_card,76777
1,boleto,19781
2,voucher,5772
3,debit_card,1528
4,desconocido,28


### R05 · Fechas inconsistentes (paso 9)

Un pedido sigue el orden compra → aprobación → entrega al transportista → entrega al cliente, y la fecha estimada no
puede ser anterior a la compra. Aquí se buscan los que no lo cumplen.

**Decisión.** Hay dos niveles:

- **Graves → cuarentena.** Entrega al cliente o fecha estimada antes de la compra, aprobación antes de la compra, o
  compra fuera de 2016–2018. Son imposibles, y como no se sabe cuál de las fechas está mal, no se pueden corregir.
- **Leves → conservar y marcar** (`fecha_inconsistente`). Estos vienen en el dataset original:
  - 166 pedidos entregados al transportista antes de la compra. La mediana del desfase es 1 hora y el 90 % está dentro
    de 2 horas; parece un problema de relojes entre sistemas.
  - 23 pedidos entregados al cliente antes que al transportista: la fecha del transportista se registró tarde.

En los dos casos la venta existió. Mandarlos a cuarentena por una fecha secundaria sería perder ventas reales.

*SQL: `CASE`, `DATEDIFF`.*

In [26]:
FECHAS_GRAVES = [
    ("R05 entrega al cliente anterior a la compra",
     "order_delivered_customer_date < order_purchase_timestamp"),
    ("R05 fecha estimada anterior a la compra",
     "CAST(order_estimated_delivery_date AS DATE) < CAST(order_purchase_timestamp AS DATE)"),
    ("R05 aprobación anterior a la compra",
     "order_approved_at < order_purchase_timestamp"),
    ("R05 compra fuera del periodo del dataset",
     "order_purchase_timestamp NOT BETWEEN TIMESTAMP '2016-01-01' AND TIMESTAMP '2018-12-31 23:59:59'"),
]
FECHAS_LEVES = [
    ("R05 entrega al transportista anterior a la compra",
     "order_delivered_carrier_date < order_purchase_timestamp"),
    ("R05 entrega al cliente anterior a la entrega al transportista",
     "order_delivered_customer_date < order_delivered_carrier_date"),
]

# La métrica convierte las fechas en una CTE, así funciona igual sobre RAW (texto) y CLEAN (timestamp)
cte_fechas = "SELECT " + ", ".join(f"a_fecha({c}) AS {c}" for c in COLS_FECHA_PEDIDOS) + " FROM pedidos_{capa}"
todas = " OR ".join(f"({cond})" for _, cond in FECHAS_GRAVES + FECHAS_LEVES)
METRICAS["R05"] = [("pedidos", f"WITH p AS ({cte_fechas}) SELECT count(*), count(*) FILTER (WHERE {todas}) FROM p")]
display(medir_regla("R05", "antes"))

display(ver(" UNION ALL ".join(
    f"SELECT '{texto}' AS verificacion, '{politica}' AS politica, count(*) AS pedidos FROM pedidos_stg WHERE {cond}"
    for lista, politica in ((FECHAS_GRAVES, "cuarentena"), (FECHAS_LEVES, "marcar"))
    for texto, cond in lista)))

# Magnitud: ¿cuántas horas antes? (los casos leves se miden sin los graves)
ver("""
    SELECT 'transportista antes de la compra (leve)' AS caso, count(*) AS pedidos,
           median(datediff('hour', order_delivered_carrier_date, order_purchase_timestamp)) AS mediana_horas,
           quantile_cont(datediff('hour', order_delivered_carrier_date, order_purchase_timestamp), 0.9) AS p90_horas,
           max(datediff('hour', order_delivered_carrier_date, order_purchase_timestamp)) AS max_horas
    FROM pedidos_stg
    WHERE order_delivered_carrier_date < order_purchase_timestamp
      AND NOT coalesce(order_delivered_customer_date < order_purchase_timestamp, false)
    UNION ALL
    SELECT 'cliente antes que transportista (leve)', count(*),
           median(datediff('hour', order_delivered_customer_date, order_delivered_carrier_date)),
           quantile_cont(datediff('hour', order_delivered_customer_date, order_delivered_carrier_date), 0.9),
           max(datediff('hour', order_delivered_customer_date, order_delivered_carrier_date))
    FROM pedidos_stg
    WHERE order_delivered_customer_date < order_delivered_carrier_date
      AND NOT coalesce(order_delivered_customer_date < order_purchase_timestamp, false)
    UNION ALL
    SELECT 'cliente antes de la compra (grave)', count(*),
           median(datediff('hour', order_delivered_customer_date, order_purchase_timestamp)),
           quantile_cont(datediff('hour', order_delivered_customer_date, order_purchase_timestamp), 0.9),
           max(datediff('hour', order_delivered_customer_date, order_purchase_timestamp))
    FROM pedidos_stg WHERE order_delivered_customer_date < order_purchase_timestamp
""")

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R05 Fechas inconsistentes,pedidos,antes,99521,249,0.25


,verificacion,politica,pedidos
0,R05 entrega al cliente anterior a la compra,cuarentena,39
1,R05 fecha estimada anterior a la compra,cuarentena,20
2,R05 aprobación anterior a la compra,cuarentena,0
3,R05 compra fuera del periodo del dataset,cuarentena,0
4,R05 entrega al transportista anterior a la compra,marcar,166
5,R05 entrega al cliente anterior a la entrega al transportista,marcar,62


,caso,pedidos,mediana_horas,p90_horas,max_horas
0,transportista antes de la compra (leve),166,1.0,2.0,4109
1,cliente antes que transportista (leve),23,40.0,166.4,386
2,cliente antes de la compra (grave),39,120.0,120.0,120


In [27]:
for motivo, cond in FECHAS_GRAVES:
    marcar_cuarentena("pedidos_stg", motivo, cond)
for texto, cond in FECHAS_LEVES:
    marcar_observacion("pedidos_stg", "fecha_inconsistente", texto, cond)

  cuarentena  pedidos_stg        39  R05 entrega al cliente anterior a la compra
  cuarentena  pedidos_stg        20  R05 fecha estimada anterior a la compra
  cuarentena  pedidos_stg         0  R05 aprobación anterior a la compra
  cuarentena  pedidos_stg         0  R05 compra fuera del periodo del dataset
  marcado     pedidos_stg       166  R05 entrega al transportista anterior a la compra
  marcado     pedidos_stg        62  R05 entrega al cliente anterior a la entrega al transportista


### R10 · Estados de pedido (paso 10)

A veces el estado no cuadra con las fechas: `delivered` sin fecha de entrega, `canceled` con fecha de entrega,
`processing` con fecha de envío, o directamente un estado vacío.

**Decisión: conservar y marcar** (`estado_inconsistente`, con el detalle en `observaciones`). No se sabe si el error
está en el estado o en la fecha, y borrar el pedido se llevaría también sus ítems y pagos. Los estados vacíos o que no
se reconocen pasan a `'desconocido'`.

*SQL: `CASE`, `GROUP BY`.*

In [28]:
CONTRADICCIONES_ESTADO = [
    ("R10 entregado sin fecha de entrega",
     "order_status = 'delivered' AND order_delivered_customer_date IS NULL"),
    ("R10 cancelado o no disponible con fecha de entrega",
     "order_status IN ('canceled', 'unavailable') AND order_delivered_customer_date IS NOT NULL"),
    ("R10 estado previo al envío con fecha de envío o entrega",
     "order_status IN ('created', 'approved', 'invoiced', 'processing') "
     "AND (order_delivered_carrier_date IS NOT NULL OR order_delivered_customer_date IS NOT NULL)"),
    ("R10 enviado o entregado sin fecha de transportista",
     "order_status IN ('shipped', 'delivered') AND order_delivered_carrier_date IS NULL"),
    ("R10 estado vacío o no reconocido",
     f"order_status IS NULL OR order_status NOT IN {lista_sql(ESTADOS_VALIDOS)}"),
]

cte_estado = ("SELECT normalizar_texto(order_status) AS order_status, "
              "a_fecha(order_delivered_carrier_date) AS order_delivered_carrier_date, "
              "a_fecha(order_delivered_customer_date) AS order_delivered_customer_date FROM pedidos_{capa}")
todas = " OR ".join(f"({cond})" for _, cond in CONTRADICCIONES_ESTADO)
METRICAS["R10"] = [("pedidos", f"WITH p AS ({cte_estado}) SELECT count(*), count(*) FILTER (WHERE {todas}) FROM p")]
display(medir_regla("R10", "antes"))

# Estado vs. fechas presentes
display(ver("""
    SELECT coalesce(order_status, '(vacío)') AS estado, count(*) AS pedidos,
           count(order_approved_at) AS con_aprobacion,
           count(order_delivered_carrier_date) AS con_fecha_transportista,
           count(order_delivered_customer_date) AS con_fecha_entrega
    FROM pedidos_stg GROUP BY ALL ORDER BY pedidos DESC
"""))
ver(" UNION ALL ".join(f"SELECT '{t}' AS contradiccion, count(*) AS pedidos FROM pedidos_stg WHERE {c}"
                       for t, c in CONTRADICCIONES_ESTADO))

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R10 Estados de pedido,pedidos,antes,99521,94,0.09


,estado,pedidos,con_aprobacion,con_fecha_transportista,con_fecha_entrega
0,delivered,96429,96375,96427,96391
1,shipped,1107,1107,1107,0
2,canceled,664,523,114,45
3,unavailable,609,609,0,0
4,invoiced,314,314,0,0
5,processing,301,301,0,0
6,(vacío),10,10,10,10
7,created,5,0,0,0
8,approved,2,2,0,0


,contradiccion,pedidos
0,R10 entregado sin fecha de entrega,38
1,R10 cancelado o no disponible con fecha de entrega,45
2,R10 estado previo al envío con fecha de envío o entrega,0
3,R10 enviado o entregado sin fecha de transportista,2
4,R10 estado vacío o no reconocido,10


In [29]:
for texto, cond in CONTRADICCIONES_ESTADO:
    marcar_observacion("pedidos_stg", "estado_inconsistente", texto, cond)

run(f"""
    UPDATE pedidos_stg SET order_status = 'desconocido'
    WHERE order_status IS NULL OR order_status NOT IN {lista_sql(ESTADOS_VALIDOS)}
""")
ver("SELECT order_status, count(*) AS pedidos, count(*) FILTER (WHERE estado_inconsistente) AS marcados "
    "FROM pedidos_stg GROUP BY ALL ORDER BY pedidos DESC")

  marcado     pedidos_stg        38  R10 entregado sin fecha de entrega
  marcado     pedidos_stg        45  R10 cancelado o no disponible con fecha de entrega
  marcado     pedidos_stg         0  R10 estado previo al envío con fecha de envío o entrega
  marcado     pedidos_stg         2  R10 enviado o entregado sin fecha de transportista
  marcado     pedidos_stg        10  R10 estado vacío o no reconocido


,order_status,pedidos,marcados
0,delivered,96429,39
1,shipped,1107,0
2,canceled,664,45
3,unavailable,609,0
4,invoiced,314,0
5,processing,301,0
6,desconocido,10,10
7,created,5,0
8,approved,2,0


### R06 · Precios inválidos (paso 11)

Precios negativos o en cero, fletes negativos y pagos negativos.

**Decisión.**

- `price <= 0`, `freight_value < 0` o `payment_value < 0` van a cuarentena. No se usa `ABS()` porque no se sabe si el
  signo es un error de tipeo, una devolución o un ajuste, y tampoco tiene sentido inventar un precio.
- Un flete de 0 es válido: es envío gratis, y pasa en el dataset original.
- Un pago de 0 se conserva y se marca: en el original son vouchers o pagos `not_defined` sin saldo.

*SQL: `WHERE`, `CASE`.*

In [30]:
METRICAS["R06"] = [
    ("items", "SELECT count(*), count(*) FILTER (WHERE a_numero(price) <= 0 OR a_numero(freight_value) < 0) FROM items_{capa}"),
    ("pagos", "SELECT count(*), count(*) FILTER (WHERE a_numero(payment_value) < 0) FROM pagos_{capa}"),
]
display(medir_regla("R06", "antes"))

ver("""
    SELECT 'items.price' AS campo, count(*) FILTER (WHERE price < 0) AS negativos,
           count(*) FILTER (WHERE price = 0) AS ceros, min(price) AS minimo, max(price) AS maximo
    FROM items_stg
    UNION ALL
    SELECT 'items.freight_value', count(*) FILTER (WHERE freight_value < 0),
           count(*) FILTER (WHERE freight_value = 0), min(freight_value), max(freight_value)
    FROM items_stg
    UNION ALL
    SELECT 'pagos.payment_value', count(*) FILTER (WHERE payment_value < 0),
           count(*) FILTER (WHERE payment_value = 0), min(payment_value), max(payment_value)
    FROM pagos_stg
""")

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R06 Precios inválidos,items,antes,112730,90,0.08
1,R06 Precios inválidos,pagos,antes,103926,0,0.00


,campo,negativos,ceros,minimo,maximo
0,items.price,30,30,-207.90,6735.00
1,items.freight_value,30,383,-177.98,409.68
2,pagos.payment_value,0,9,0.00,13664.08


In [31]:
marcar_cuarentena("items_stg", "R06 precio menor o igual a cero", "price <= 0")
marcar_cuarentena("items_stg", "R06 flete negativo", "freight_value < 0")
marcar_cuarentena("pagos_stg", "R06 monto de pago negativo", "payment_value < 0")
marcar_observacion("pagos_stg", "pago_cero", "R06 pago con valor cero", "payment_value = 0")

ver("SELECT payment_type, count(*) AS pagos_en_cero FROM pagos_stg WHERE pago_cero GROUP BY ALL")

  cuarentena  items_stg          60  R06 precio menor o igual a cero
  cuarentena  items_stg          30  R06 flete negativo
  cuarentena  pagos_stg           0  R06 monto de pago negativo
  marcado     pagos_stg           9  R06 pago con valor cero


,payment_type,pagos_en_cero
0,desconocido,3
1,voucher,6


### R09 · Integridad referencial (paso 12)

Se buscan huérfanos: ítems o pagos de pedidos que no existen, ítems de productos que no existen y pedidos de clientes
que no existen.

**Decisión.**

- Los huérfanos van a cuarentena: no se pueden asociar a nada y romperían los `JOIN`.
- Si un pedido terminó en cuarentena (por R04, R05 o aquí mismo), sus ítems y pagos lo siguen, con el motivo
  `R09 pedido en cuarentena`. Sin esta cascada, CLEAN tendría ítems de pedidos que no están.
- Los pedidos sin ítems o sin pagos válidos se quedan, marcados. En el original hay 775 pedidos sin ítems, casi todos
  cancelados o no disponibles, y siguen siendo pedidos.

*SQL: `LEFT JOIN`, `NOT EXISTS`.*

In [32]:
METRICAS["R09"] = [
    ("pedidos", "SELECT count(*), count(*) FILTER (WHERE c.customer_id IS NULL) FROM pedidos_{capa} AS p "
                "LEFT JOIN (SELECT DISTINCT customer_id FROM clientes_{capa}) AS c USING (customer_id)"),
    ("items", "SELECT count(*), count(*) FILTER (WHERE "
              "NOT EXISTS (SELECT 1 FROM pedidos_{capa} p WHERE p.order_id = i.order_id) "
              "OR NOT EXISTS (SELECT 1 FROM productos_{capa} pr WHERE pr.product_id = i.product_id)) "
              "FROM items_{capa} AS i"),
    ("pagos", "SELECT count(*), count(*) FILTER (WHERE NOT EXISTS "
              "(SELECT 1 FROM pedidos_{capa} p WHERE p.order_id = g.order_id)) FROM pagos_{capa} AS g"),
]
display(medir_regla("R09", "antes"))

ver("""
    SELECT 'pedidos -> clientes' AS relacion, count(*) AS registros, count(*) FILTER (WHERE c.customer_id IS NULL) AS huerfanos
    FROM pedidos_stg p LEFT JOIN clientes_stg c USING (customer_id)
    UNION ALL
    SELECT 'items -> pedidos', count(*), count(*) FILTER (WHERE p.order_id IS NULL)
    FROM items_stg i LEFT JOIN pedidos_stg p USING (order_id)
    UNION ALL
    SELECT 'items -> productos', count(*), count(*) FILTER (WHERE pr.product_id IS NULL)
    FROM items_stg i LEFT JOIN productos_stg pr USING (product_id)
    UNION ALL
    SELECT 'pagos -> pedidos', count(*), count(*) FILTER (WHERE p.order_id IS NULL)
    FROM pagos_stg g LEFT JOIN pedidos_stg p USING (order_id)
    UNION ALL
    SELECT 'pedidos sin ítems (informativo)', count(*), count(*) FILTER (WHERE NOT EXISTS
           (SELECT 1 FROM items_stg i WHERE i.order_id = p.order_id))
    FROM pedidos_stg p
""")

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R09 Integridad referencial,items,antes,112730,50,0.04
1,R09 Integridad referencial,pagos,antes,103926,20,0.02
2,R09 Integridad referencial,pedidos,antes,99521,30,0.03


,relacion,registros,huerfanos
0,pedidos -> clientes,99441,30
1,items -> pedidos,112650,20
2,items -> productos,112650,30
3,pagos -> pedidos,103886,20
4,pedidos sin ítems (informativo),99441,791


In [33]:
# 1) Pedidos de clientes inexistentes
marcar_cuarentena("pedidos_stg", "R09 cliente inexistente",
                  "NOT EXISTS (SELECT 1 FROM clientes_stg c WHERE c.customer_id = pedidos_stg.customer_id)")

# 2) Ítems y pagos: huérfanos y cascada de pedidos en cuarentena
for hija in ("items", "pagos"):
    marcar_cuarentena(f"{hija}_stg", "R09 pedido inexistente",
                      f"NOT EXISTS (SELECT 1 FROM pedidos_stg p WHERE p.order_id = {hija}_stg.order_id)")
    marcar_cuarentena(f"{hija}_stg", "R09 pedido en cuarentena",
                      f"EXISTS (SELECT 1 FROM pedidos_stg p WHERE p.order_id = {hija}_stg.order_id "
                      f"AND p.motivo_cuarentena IS NOT NULL)")
marcar_cuarentena("items_stg", "R09 producto inexistente",
                  "NOT EXISTS (SELECT 1 FROM productos_stg pr WHERE pr.product_id = items_stg.product_id)")

# 3) Pedidos válidos sin ítems o sin pagos válidos: se conservan marcados
marcar_observacion("pedidos_stg", "sin_items", "R09 pedido sin ítems válidos",
                   "motivo_cuarentena IS NULL AND NOT EXISTS (SELECT 1 FROM items_stg i "
                   "WHERE i.order_id = pedidos_stg.order_id AND i.motivo_cuarentena IS NULL)")
marcar_observacion("pedidos_stg", "sin_pagos", "R09 pedido sin pagos válidos",
                   "motivo_cuarentena IS NULL AND NOT EXISTS (SELECT 1 FROM pagos_stg g "
                   "WHERE g.order_id = pedidos_stg.order_id AND g.motivo_cuarentena IS NULL)")

# Pedidos sin ítems válidos por estado: casi todos son cancelados o no disponibles
ver("""
    SELECT order_status, count(*) AS pedidos_sin_items
    FROM pedidos_stg WHERE sin_items
    GROUP BY ALL ORDER BY pedidos_sin_items DESC
""")

  cuarentena  pedidos_stg        30  R09 cliente inexistente
  cuarentena  items_stg          20  R09 pedido inexistente
  cuarentena  items_stg         119  R09 pedido en cuarentena
  cuarentena  pagos_stg          20  R09 pedido inexistente
  cuarentena  pagos_stg         103  R09 pedido en cuarentena
  cuarentena  items_stg          30  R09 producto inexistente
  marcado     pedidos_stg       899  R09 pedido sin ítems válidos
  marcado     pedidos_stg        28  R09 pedido sin pagos válidos


,order_status,pedidos_sin_items
0,unavailable,603
1,canceled,164
2,delivered,121
3,created,5
4,shipped,4
5,invoiced,2


### R07 · Valores atípicos con IQR (paso 13)

Para precio, flete y pago se usa el rango intercuartílico: `IQR = Q3 − Q1`, y es atípico lo que queda por debajo de
`Q1 − 1.5·IQR` o por encima de `Q3 + 1.5·IQR`.

**Decisión: conservar y marcar** (`atipico_precio`, `atipico_flete`, `atipico_pago`). Que un valor sea atípico no lo
convierte en un error: en una tienda online hay productos caros de verdad y fletes altos a zonas lejanas, y sacarlos
bajaría artificialmente las ventas y el ticket promedio. Los límites se calculan solo con registros válidos (después
de R06) y quedan guardados en `limites_iqr`.

*SQL: `QUANTILE_CONT`.*

In [34]:
def plantilla_iqr(tabla, col):
    return f"""
        WITH v AS (SELECT a_numero({col}) AS x FROM {tabla}_{{capa}} WHERE a_numero({col}) IS NOT NULL),
             l AS (SELECT quantile_cont(x, 0.25) AS q1, quantile_cont(x, 0.75) AS q3 FROM v)
        SELECT count(*), count(*) FILTER (WHERE x < q1 - 1.5 * (q3 - q1) OR x > q3 + 1.5 * (q3 - q1))
        FROM v, l"""

METRICAS["R07"] = [
    ("items.price", plantilla_iqr("items", "price")),
    ("items.freight_value", plantilla_iqr("items", "freight_value")),
    ("pagos.payment_value", plantilla_iqr("pagos", "payment_value")),
]
display(medir_regla("R07", "antes"))

run("""
CREATE OR REPLACE TABLE limites_iqr AS
WITH valores AS (
    SELECT 'price' AS variable, CAST(price AS DOUBLE) AS valor FROM items_stg WHERE motivo_cuarentena IS NULL
    UNION ALL
    SELECT 'freight_value', CAST(freight_value AS DOUBLE) FROM items_stg WHERE motivo_cuarentena IS NULL
    UNION ALL
    SELECT 'payment_value', CAST(payment_value AS DOUBLE) FROM pagos_stg WHERE motivo_cuarentena IS NULL
),
cuartiles AS (
    SELECT variable, count(*) AS n,
           quantile_cont(valor, 0.25) AS q1,
           quantile_cont(valor, 0.75) AS q3
    FROM valores GROUP BY variable
)
SELECT variable, n, q1, q3, q3 - q1 AS iqr,
       q1 - 1.5 * (q3 - q1) AS limite_inferior,
       q3 + 1.5 * (q3 - q1) AS limite_superior
FROM cuartiles
""")
ver("SELECT * FROM limites_iqr ORDER BY variable")

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R07 Valores atípicos (IQR),items.freight_value,antes,112730,12167,10.79
1,R07 Valores atípicos (IQR),items.price,antes,112718,8443,7.49
2,R07 Valores atípicos (IQR),pagos.payment_value,antes,103916,7990,7.69


,variable,n,q1,q3,iqr,limite_inferior,limite_superior
0,freight_value,112379,13.08,21.15,8.07,0.975,33.255
1,payment_value,103753,56.79,171.79,115.00,-115.710,344.290
2,price,112379,39.90,134.90,95.00,-102.600,277.400


In [35]:
def fuera_de_rango(col, variable):
    return (f"motivo_cuarentena IS NULL AND ("
            f"{col} < (SELECT limite_inferior FROM limites_iqr WHERE variable = '{variable}') OR "
            f"{col} > (SELECT limite_superior FROM limites_iqr WHERE variable = '{variable}'))")

marcar_observacion("items_stg", "atipico_precio", "R07 precio atípico (IQR)", fuera_de_rango("price", "price"))
marcar_observacion("items_stg", "atipico_flete", "R07 flete atípico (IQR)", fuera_de_rango("freight_value", "freight_value"))
marcar_observacion("pagos_stg", "atipico_pago", "R07 pago atípico (IQR)", fuera_de_rango("payment_value", "payment_value"))

# ¿Son errores? Las categorías con más precios atípicos son de productos caros por naturaleza
ver("""
    SELECT pr.product_category_name AS categoria, count(*) AS items_con_precio_atipico,
           round(median(i.price), 2) AS mediana_precio, max(i.price) AS precio_maximo
    FROM items_stg AS i JOIN productos_stg AS pr USING (product_id)
    WHERE i.atipico_precio
    GROUP BY ALL ORDER BY items_con_precio_atipico DESC
    LIMIT 8
""")

  marcado     items_stg        8412  R07 precio atípico (IQR)
  marcado     items_stg       12107  R07 flete atípico (IQR)
  marcado     pagos_stg        7976  R07 pago atípico (IQR)


,categoria,items_con_precio_atipico,mediana_precio,precio_maximo
0,beleza_saude,1155,370.00,3124.00
1,relogios_presentes,972,559.45,3999.90
2,esporte_lazer,485,389.00,4059.00
3,automotivo,425,429.90,2258.00
4,cool_stuff,390,539.99,3109.99
5,informatica_acessorios,375,439.90,3699.99
6,brinquedos,352,349.99,1699.99
7,perfumaria,333,339.90,689.90


### R12 · Pagos frente a ítems (paso 14)

Lo pagado en un pedido debería coincidir con la suma de `price + freight_value` de sus ítems. Como un pedido se puede
pagar en varias partes (tarjeta y voucher, por ejemplo), se compara el total pagado por pedido y no cada pago.

**Decisión: conservar y marcar** los pedidos donde la diferencia pasa de 1,00 BRL (`pago_inconsistente`). Puede haber
razones legítimas, como intereses por cuotas, vouchers o ítems que quedaron en cuarentena. El detalle de cada pedido
está en `conciliacion_pagos`.

*SQL: `SUM`, `ABS`, `JOIN`.*

In [36]:
METRICAS["R12"] = [
    ("pedidos", f"""
        WITH it AS (SELECT order_id, sum(a_numero(price) + a_numero(freight_value)) AS total_items
                    FROM items_{{capa}} GROUP BY order_id),
             pg AS (SELECT order_id, sum(a_numero(payment_value)) AS total_pagado
                    FROM pagos_{{capa}} GROUP BY order_id)
        SELECT count(*), count(*) FILTER (WHERE abs(total_pagado - total_items) > {TOLERANCIA_PAGO})
        FROM it JOIN pg USING (order_id)"""),
]
display(medir_regla("R12", "antes"))

run(f"""
CREATE OR REPLACE TABLE conciliacion_pagos AS
WITH it AS (
    SELECT order_id, count(*) AS num_items, sum(price + freight_value) AS total_items
    FROM items_stg WHERE motivo_cuarentena IS NULL GROUP BY order_id
),
pg AS (
    SELECT order_id, count(*) AS num_pagos, count(DISTINCT payment_type) AS num_metodos,
           max(payment_installments) AS max_cuotas, sum(payment_value) AS total_pagado
    FROM pagos_stg WHERE motivo_cuarentena IS NULL GROUP BY order_id
)
SELECT p.order_id, p.order_status,
       coalesce(it.num_items, 0)    AS num_items,
       coalesce(it.total_items, 0)  AS total_items,
       coalesce(pg.num_pagos, 0)    AS num_pagos,
       coalesce(pg.num_metodos, 0)  AS num_metodos,
       pg.max_cuotas,
       coalesce(pg.total_pagado, 0) AS total_pagado,
       coalesce(pg.total_pagado, 0) - coalesce(it.total_items, 0) AS diferencia,
       CASE
           WHEN it.order_id IS NULL OR pg.order_id IS NULL THEN 'sin ítems o sin pagos'
           WHEN abs(diferencia) <= {TOLERANCIA_PAGO} THEN 'cuadra'
           WHEN diferencia > 0 THEN 'pagado de más'
           ELSE 'pagado de menos'
       END AS resultado
FROM pedidos_stg AS p
LEFT JOIN it USING (order_id)
LEFT JOIN pg USING (order_id)
WHERE p.motivo_cuarentena IS NULL
""")

ver("""
    SELECT resultado, count(*) AS pedidos,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct,
           count(*) FILTER (WHERE num_pagos > 1) AS con_varios_pagos,
           count(*) FILTER (WHERE max_cuotas > 1) AS pagados_en_cuotas,
           round(median(diferencia), 2) AS mediana_diferencia
    FROM conciliacion_pagos GROUP BY ALL ORDER BY pedidos DESC
""")

,nombre_regla,tabla_evaluada,etapa,registros_evaluados,registros_invalidos,porcentaje_error
0,R12 Pagos vs ítems,pedidos,antes,98630,392,0.4


,resultado,pedidos,pct,con_varios_pagos,pagados_en_cuotas,mediana_diferencia
0,cuadra,98134,98.79,2917,50451,0.00
1,sin ítems o sin pagos,927,0.93,28,393,102.66
2,pagado de más,259,0.26,11,243,8.78
3,pagado de menos,20,0.02,0,8,-10.20


In [37]:
marcar_observacion("pedidos_stg", "pago_inconsistente", "R12 pagos no cuadran con ítems + flete",
                   "order_id IN (SELECT order_id FROM conciliacion_pagos "
                   "WHERE resultado IN ('pagado de más', 'pagado de menos'))")

# Las mayores diferencias
ver("""
    SELECT order_id, order_status, num_items, total_items, num_pagos, max_cuotas, total_pagado, diferencia, resultado
    FROM conciliacion_pagos
    WHERE resultado IN ('pagado de más', 'pagado de menos')
    ORDER BY abs(diferencia) DESC
    LIMIT 8
""")

  marcado     pedidos_stg       279  R12 pagos no cuadran con ítems + flete


,order_id,order_status,num_items,total_items,num_pagos,max_cuotas,total_pagado,diferencia,resultado
0,cd9fc86d944e6e7e6845deab57fdb8bc,delivered,1,60.24,1,8,337.09,276.85,pagado de más
1,08bdbdeaf1626720051193300f93806e,delivered,4,473.12,1,1,199.44,-273.68,pagado de menos
2,47dc3a1de5a0aec4323cdade0588655c,delivered,1,425.12,1,5,671.85,246.73,pagado de más
3,a688c7001406d268e4d8779ab25573a1,delivered,1,281.17,1,1,522.34,241.17,pagado de más
4,3e79636f6031f6b4f05b671928369588,delivered,1,200.32,1,6,400.64,200.32,pagado de más
5,ce6d150fb29ada17d2082f4847107665,delivered,1,1403.66,1,10,1586.47,182.81,pagado de más
6,dea9b7dcae5affc919947e72bcc1e786,delivered,1,136.99,1,1,273.98,136.99,pagado de más
7,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,6,287.91,1,10,406.92,119.01,pagado de más


## 4. Capa CLEAN y cuarentena

Con las reglas aplicadas, cada tabla de staging se parte en dos: `*_clean`, con los registros sin motivo de cuarentena
(y sus marcas), y `*_cuarentena`, con lo que no se pudo arreglar y el porqué. Nada se pierde: lo apartado se puede
revisar y, si se corrige en el origen, volver a cargar. R15 verifica que `raw = clean + cuarentena + duplicados eliminados`.

In [38]:
for t in TABLAS:
    run(f"CREATE OR REPLACE TABLE {t}_clean AS "
        f"SELECT * EXCLUDE (motivo_cuarentena) FROM {t}_stg WHERE motivo_cuarentena IS NULL")
    run(f"CREATE OR REPLACE TABLE {t}_cuarentena AS "
        f"SELECT * FROM {t}_stg WHERE motivo_cuarentena IS NOT NULL")

ver(" UNION ALL ".join(f"""
    SELECT '{t}' AS tabla,
           (SELECT count(*) FROM {t}_raw) AS raw,
           (SELECT count(*) FROM {t}_raw) - (SELECT count(*) FROM {t}_stg) AS duplicados_eliminados,
           (SELECT count(*) FROM {t}_cuarentena) AS cuarentena,
           (SELECT count(*) FROM {t}_clean) AS clean,
           round(100.0 * (SELECT count(*) FROM {t}_clean) / (SELECT count(*) FROM {t}_raw), 2) AS pct_conservado
""" for t in TABLAS))

,tabla,raw,duplicados_eliminados,cuarentena,clean,pct_conservado
0,pedidos,99521,80,101,99340,99.82
1,items,112730,80,271,112379,99.69
2,clientes,99491,50,0,99441,99.95
3,pagos,103926,40,133,103753,99.83
4,productos,32981,30,0,32951,99.91


In [39]:
# Motivos de cuarentena (un registro puede tener más de uno)
motivos = " UNION ALL ".join(
    f"SELECT '{t}' AS tabla, unnest(string_split(motivo_cuarentena, '; ')) AS motivo FROM {t}_cuarentena"
    for t in TABLAS)
run(f"""
    CREATE OR REPLACE TABLE resumen_cuarentena AS
    SELECT tabla, motivo, count(*) AS registros FROM ({motivos}) GROUP BY ALL
""")
display(ver("SELECT * FROM resumen_cuarentena ORDER BY tabla, registros DESC"))

# Problemas conservados y marcados en CLEAN
observaciones = " UNION ALL ".join(
    f"SELECT '{t}' AS tabla, unnest(string_split(observaciones, '; ')) AS observacion FROM {t}_clean"
    for t in TABLAS)
ver(f"SELECT tabla, observacion, count(*) AS registros FROM ({observaciones}) GROUP BY ALL ORDER BY tabla, registros DESC")

,tabla,motivo,registros
0,items,R09 pedido en cuarentena,119
1,items,R06 precio menor o igual a cero,60
2,items,R09 producto inexistente,30
3,items,R06 flete negativo,30
4,items,R09 pedido inexistente,20
5,items,"R04 price no convertible a DECIMAL(10,2)",12
6,pagos,R09 pedido en cuarentena,103
7,pagos,R09 pedido inexistente,20
8,pagos,R02 campo crítico nulo,10
9,pedidos,R05 entrega al cliente anterior a la compra,39


,tabla,observacion,registros
0,clientes,R11 código postal completado con LPAD,289
1,clientes,R02 ciudad imputada desde el código postal,40
2,clientes,R11 código postal inválido,15
3,items,R07 flete atípico (IQR),12107
4,items,R07 precio atípico (IQR),8412
5,pagos,R07 pago atípico (IQR),7976
6,pagos,R13 método de pago desconocido,28
7,pagos,R06 pago con valor cero,9
8,pedidos,R09 pedido sin ítems válidos,899
9,pedidos,R12 pagos no cuadran con ítems + flete,279


## 5. Validación antes y después

Se vuelven a correr las mismas plantillas de métrica, ahora sobre CLEAN, y se comparan con RAW. Se agrega también la
completitud (`M01`), que es el porcentaje de celdas sin dato.

En la columna `resultado`, `resuelto` significa que el problema ya no aparece en CLEAN. `reducido` y
`conservado y marcado` significan que quedan casos a propósito, porque la regla los conserva con una marca: atípicos,
estados contradictorios, pagos que no cuadran, desfases leves de fechas y métodos desconocidos.

In [40]:
METRICAS["M01"] = []
for t in TABLAS:
    cols = columnas(f"{t}_raw")
    sin_dato = " + ".join(f'count(*) FILTER (WHERE "{c}" IS NULL OR es_vacio("{c}"))' for c in cols)
    METRICAS["M01"].append((t, f"SELECT count(*) * {len(cols)}, {sin_dato} FROM {t}_{{capa}}"))
medir_regla("M01", "antes")

for regla in METRICAS:
    medir_regla(regla, "despues")

run("""
CREATE OR REPLACE TABLE comparacion_antes_despues AS
SELECT a.nombre_regla AS regla, a.tabla_evaluada AS tabla,
       a.registros_evaluados AS evaluados_antes,  a.registros_invalidos AS invalidos_antes,  a.porcentaje_error AS pct_antes,
       d.registros_evaluados AS evaluados_despues, d.registros_invalidos AS invalidos_despues, d.porcentaje_error AS pct_despues,
       CASE
           WHEN d.registros_invalidos = 0 THEN 'resuelto'
           WHEN d.registros_invalidos < a.registros_invalidos THEN 'reducido'
           ELSE 'conservado y marcado'
       END AS resultado,
       a.politica
FROM quality_report AS a
JOIN quality_report AS d
  ON d.nombre_regla = a.nombre_regla AND d.tabla_evaluada = a.tabla_evaluada AND d.etapa = 'despues'
WHERE a.etapa = 'antes'
""")
ver("SELECT * EXCLUDE (politica) FROM comparacion_antes_despues ORDER BY regla, tabla")

,regla,tabla,evaluados_antes,invalidos_antes,pct_antes,evaluados_despues,invalidos_despues,pct_despues,resultado
0,M01 Completitud,clientes,497455,40,0.01,497205,15,0.00,reducido
1,M01 Completitud,items,789110,0,0.00,786653,0,0.00,resuelto
2,M01 Completitud,pagos,519630,20,0.00,518765,0,0.00,resuelto
3,M01 Completitud,pedidos,796168,4991,0.63,794720,4975,0.63,reducido
4,M01 Completitud,productos,296829,2486,0.84,296559,1848,0.62,reducido
5,R01 Duplicados,clientes,99491,50,0.05,99441,0,0.00,resuelto
6,R01 Duplicados,items,112730,80,0.07,112379,0,0.00,resuelto
7,R01 Duplicados,pagos,103926,40,0.04,103753,0,0.00,resuelto
8,R01 Duplicados,pedidos,99521,80,0.08,99340,0,0.00,resuelto
9,R01 Duplicados,productos,32981,30,0.09,32951,0,0.00,resuelto


### Métricas mínimas del enunciado

In [41]:
ver("""
    SELECT CASE
               WHEN regla LIKE 'R01%' THEN 'Duplicados'
               WHEN regla LIKE 'R02%' THEN 'Nulos críticos'
               WHEN regla LIKE 'R05%' THEN 'Fechas inconsistentes'
               WHEN regla LIKE 'R06%' THEN 'Precios inválidos'
               WHEN regla LIKE 'R09%' THEN 'Registros huérfanos'
               WHEN regla LIKE 'M01%' THEN 'Completitud (% celdas con dato)'
           END AS metrica,
           tabla,
           CASE WHEN regla LIKE 'M01%' THEN round(100 - pct_antes, 2) ELSE invalidos_antes END AS antes,
           CASE WHEN regla LIKE 'M01%' THEN round(100 - pct_despues, 2) ELSE invalidos_despues END AS despues
    FROM comparacion_antes_despues
    WHERE regla SIMILAR TO '(R01|R02|R05|R06|R09|M01).*'
    ORDER BY metrica, tabla
""")

,metrica,tabla,antes,despues
0,Completitud (% celdas con dato),clientes,99.99,100.00
1,Completitud (% celdas con dato),items,100.00,100.00
2,Completitud (% celdas con dato),pagos,100.00,100.00
3,Completitud (% celdas con dato),pedidos,99.37,99.37
4,Completitud (% celdas con dato),productos,99.16,99.38
5,Duplicados,clientes,50.00,0.00
6,Duplicados,items,80.00,0.00
7,Duplicados,pagos,40.00,0.00
8,Duplicados,pedidos,80.00,0.00
9,Duplicados,productos,30.00,0.00


### R15 · Validación final

Un último control sobre CLEAN. Cada verificación cuenta violaciones y tiene que dar 0; si alguna falla, el `assert`
detiene el notebook. Los resultados quedan en `validacion_final`.

| Grupo | Qué revisa |
|---|---|
| Unicidad | cada clave de negocio aparece una sola vez |
| Integridad | no hay ítems, pagos ni pedidos huérfanos |
| Tipos | cada columna tiene el tipo esperado |
| Errores críticos | sin nulos críticos, vacíos, precios inválidos ni fechas imposibles; valores dentro de su dominio |
| Balance | `raw = clean + cuarentena + duplicados`: no se perdió nada sin explicación |

*SQL: `GROUP BY`, `HAVING`, `NOT EXISTS`.*

In [42]:
verificaciones = []
def verificar(grupo, objeto, sql_conteo):
    verificaciones.append(f"SELECT '{grupo}' AS grupo, '{objeto}' AS verificacion, ({sql_conteo}) AS violaciones")

for t, k in CLAVES_NEGOCIO.items():
    clave = ", ".join(k)
    verificar("Unicidad", f"{t}_clean: clave ({clave}) única",
              f"SELECT count(*) FROM (SELECT {clave} FROM {t}_clean GROUP BY ALL HAVING count(*) > 1)")
    verificar("Errores críticos", f"{t}_clean: campos críticos sin NULL",
              f"SELECT count(*) FROM {t}_clean WHERE {cualquiera('{c} IS NULL', CRITICAS[t])}")
    verificar("Errores críticos", f"{t}_clean: sin cadenas vacías",
              f"SELECT count(*) FROM {t}_clean WHERE {cualquiera('es_vacio({c})', columnas(t + '_raw'))}")
    verificar("Balance", f"{t}: raw = clean + cuarentena + duplicados",
              f"SELECT abs((SELECT count(*) FROM {t}_raw) - (SELECT count(*) FROM {t}_clean) "
              f"- (SELECT count(*) FROM {t}_cuarentena) - (SELECT registros_invalidos FROM quality_report "
              f"WHERE nombre_regla LIKE 'R01 %' AND tabla_evaluada = '{t}' AND etapa = 'antes'))")

verificar("Integridad", "items_clean -> pedidos_clean",
          "SELECT count(*) FROM items_clean i WHERE NOT EXISTS (SELECT 1 FROM pedidos_clean p WHERE p.order_id = i.order_id)")
verificar("Integridad", "items_clean -> productos_clean",
          "SELECT count(*) FROM items_clean i WHERE NOT EXISTS (SELECT 1 FROM productos_clean p WHERE p.product_id = i.product_id)")
verificar("Integridad", "pedidos_clean -> clientes_clean",
          "SELECT count(*) FROM pedidos_clean p WHERE NOT EXISTS (SELECT 1 FROM clientes_clean c WHERE c.customer_id = p.customer_id)")
verificar("Integridad", "pagos_clean -> pedidos_clean",
          "SELECT count(*) FROM pagos_clean g WHERE NOT EXISTS (SELECT 1 FROM pedidos_clean p WHERE p.order_id = g.order_id)")

for t, cols in CONVERSIONES.items():
    for c, tipo in cols.items():
        verificar("Tipos", f"{t}_clean.{c} es {tipo}",
                  f"SELECT count(*) FROM information_schema.columns WHERE table_name = '{t}_clean' "
                  f"AND column_name = '{c}' AND data_type <> '{tipo}'")

verificar("Errores críticos", "items_clean: precio > 0 y flete >= 0",
          "SELECT count(*) FROM items_clean WHERE price <= 0 OR freight_value < 0")
verificar("Errores críticos", "pagos_clean: monto >= 0",
          "SELECT count(*) FROM pagos_clean WHERE payment_value < 0")
verificar("Errores críticos", "pedidos_clean: sin secuencias de fechas imposibles",
          "SELECT count(*) FROM pedidos_clean WHERE " + " OR ".join(f"({c})" for _, c in FECHAS_GRAVES))
verificar("Errores críticos", "pedidos_clean: estado dentro del dominio",
          f"SELECT count(*) FROM pedidos_clean WHERE order_status NOT IN {lista_sql(ESTADOS_VALIDOS + ['desconocido'])}")
verificar("Errores críticos", "pagos_clean: método dentro del dominio",
          f"SELECT count(*) FROM pagos_clean WHERE payment_type NOT IN {lista_sql(METODOS_VALIDOS + ['desconocido'])}")
verificar("Errores críticos", "clientes_clean: CP NULL o de 5 dígitos",
          "SELECT count(*) FROM clientes_clean WHERE NOT regexp_matches(customer_zip_code_prefix, '^[0-9]{5}$')")
verificar("Errores críticos", "clientes_clean: ciudad y UF estandarizadas",
          "SELECT count(*) FROM clientes_clean WHERE customer_city <> normalizar_texto(customer_city) "
          "OR customer_state <> upper(trim(customer_state))")
verificar("Errores críticos", "productos_clean: categoría del catálogo y traducida",
          "SELECT count(*) FROM productos_clean WHERE product_category_name NOT IN "
          "(SELECT categoria_estandar FROM catalogo_categorias) OR product_category_name_english IS NULL")

run("CREATE OR REPLACE TABLE validacion_final AS "
    "SELECT *, CASE WHEN violaciones = 0 THEN 'OK' ELSE 'FALLA' END AS resultado FROM ("
    + " UNION ALL ".join(verificaciones) + ")")
medir("R15", "todas", "despues",
      "SELECT count(*), count(*) FILTER (WHERE resultado = 'FALLA') FROM validacion_final")

resultado_r15 = ver("SELECT * FROM validacion_final ORDER BY grupo, verificacion")
display(resultado_r15)
fallas = resultado_r15[resultado_r15.resultado == "FALLA"]
assert fallas.empty, f"Fallaron {len(fallas)} verificaciones:\n{fallas}"
print(f"Las {len(resultado_r15)} verificaciones pasaron (0 violaciones).")

,grupo,verificacion,violaciones,resultado
0,Balance,clientes: raw = clean + cuarentena + duplicados,0,OK
1,Balance,items: raw = clean + cuarentena + duplicados,0,OK
2,Balance,pagos: raw = clean + cuarentena + duplicados,0,OK
3,Balance,pedidos: raw = clean + cuarentena + duplicados,0,OK
4,Balance,productos: raw = clean + cuarentena + duplicados,0,OK
5,Errores críticos,clientes_clean: CP NULL o de 5 dígitos,0,OK
6,Errores críticos,clientes_clean: campos críticos sin NULL,0,OK
7,Errores críticos,clientes_clean: ciudad y UF estandarizadas,0,OK
8,Errores críticos,clientes_clean: sin cadenas vacías,0,OK
9,Errores críticos,items_clean: campos críticos sin NULL,0,OK


Las 51 verificaciones pasaron (0 violaciones).


## 6. Exportación

El dataset limpio se exporta a Parquet en `data/processed/` (un archivo por tabla) y los reportes a CSV en `reports/`.
La base `ecommerce.duckdb` se queda con todas las tablas.

In [43]:
for t in TABLAS:
    destino = (DIR_PROCESSED / f"{t}_clean.parquet").as_posix()
    run(f"COPY {t}_clean TO '{destino}' (FORMAT parquet)")

for tabla in ("quality_report", "comparacion_antes_despues", "validacion_final", "resumen_cuarentena", "errores_inyectados"):
    destino = (DIR_REPORTS / f"{tabla}.csv").as_posix()
    run(f"COPY (SELECT * FROM {tabla} ORDER BY ALL) TO '{destino}' (HEADER)")

# Comprobación: el Parquet tiene los mismos registros que la tabla CLEAN
ver(" UNION ALL ".join(f"""
    SELECT '{t}_clean.parquet' AS archivo,
           (SELECT count(*) FROM read_parquet('{(DIR_PROCESSED / f"{t}_clean.parquet").as_posix()}')) AS registros_parquet,
           (SELECT count(*) FROM {t}_clean) AS registros_tabla
""" for t in TABLAS))

,archivo,registros_parquet,registros_tabla
0,pedidos_clean.parquet,99340,99340
1,items_clean.parquet,112379,112379
2,clientes_clean.parquet,99441,99441
3,pagos_clean.parquet,103753,103753
4,productos_clean.parquet,32951,32951


## 7. Idempotencia

Para comprobar que correr el pipeline dos veces da lo mismo, se saca una huella de cada tabla y de los CSV de trabajo
(un `md5` del contenido ordenado), se vuelven a ejecutar todas las sentencias de `PIPELINE` y se comparan las huellas.
De `quality_report` se excluye `fecha_ejecucion`, que cambia en cada corrida.

In [44]:
TABLAS_RESULTADO = ([f"{t}_{capa}" for t in TABLAS for capa in ("raw", "clean", "cuarentena")]
                    + ["catalogo_categorias", "limites_iqr", "conciliacion_pagos", "comparacion_antes_despues",
                       "validacion_final", "resumen_cuarentena", "errores_inyectados", "quality_report"])

def huella(tabla):
    excluir = " EXCLUDE (fecha_ejecucion)" if tabla == "quality_report" else ""
    return con.execute(f"""
        SELECT count(*), coalesce(md5(string_agg(h, '' ORDER BY h)), md5(''))   -- md5('') si la tabla está vacía
        FROM (SELECT md5(CAST(t AS VARCHAR)) AS h FROM (SELECT *{excluir} FROM {tabla}) AS t)
    """).fetchone()

def huella_csv_trabajo():
    return {a: hashlib.md5((DIR_TRABAJO / a).read_bytes()).hexdigest() for a in ARCHIVOS.values()}

huellas_1 = {t: huella(t) for t in TABLAS_RESULTADO}
csv_1 = huella_csv_trabajo()

for sql in PIPELINE:          # segunda ejecución completa, mismas sentencias y mismo orden
    con.execute(sql)

huellas_2 = {t: huella(t) for t in TABLAS_RESULTADO}
csv_2 = huella_csv_trabajo()

idempotencia = pd.DataFrame([
    {"objeto": t, "registros_1": huellas_1[t][0], "registros_2": huellas_2[t][0],
     "huella_1": huellas_1[t][1][:12], "huella_2": huellas_2[t][1][:12], "identico": huellas_1[t] == huellas_2[t]}
    for t in TABLAS_RESULTADO
] + [
    {"objeto": f"data/trabajo/{a}", "registros_1": None, "registros_2": None,
     "huella_1": csv_1[a][:12], "huella_2": csv_2[a][:12], "identico": csv_1[a] == csv_2[a]}
    for a in ARCHIVOS.values()
])
display(idempotencia)
assert idempotencia.identico.all(), "El pipeline NO es idempotente"
print(f"Pipeline idempotente: {len(PIPELINE)} sentencias ejecutadas dos veces con resultado idéntico.")

,objeto,registros_1,registros_2,huella_1,huella_2,identico
0,pedidos_raw,99521.0,99521.0,ff936bd1e511,ff936bd1e511,True
1,pedidos_clean,99340.0,99340.0,e425eff226b7,e425eff226b7,True
2,pedidos_cuarentena,101.0,101.0,028405cf190f,028405cf190f,True
3,items_raw,112730.0,112730.0,9a4668d61eba,9a4668d61eba,True
4,items_clean,112379.0,112379.0,913f02e854fe,913f02e854fe,True
5,items_cuarentena,271.0,271.0,b26d76a0d78e,b26d76a0d78e,True
6,clientes_raw,99491.0,99491.0,c758bddb030b,c758bddb030b,True
7,clientes_clean,99441.0,99441.0,4acff78828e8,4acff78828e8,True
8,clientes_cuarentena,0.0,0.0,d41d8cd98f00,d41d8cd98f00,True
9,pagos_raw,103926.0,103926.0,641d05665cc0,641d05665cc0,True


Pipeline idempotente: 356 sentencias ejecutadas dos veces con resultado idéntico.


## 8. Trazabilidad

Todo el SQL ejecutado queda en `sql/pipeline_etl.sql`, en orden. Junto con `quality_report`, `errores_inyectados`, las
columnas `motivo_cuarentena` y `observaciones` y los flags, permite rastrear qué se le hizo a cada dato y por qué.

In [45]:
encabezado = (
    "-- Pipeline ETL generado por notebooks/01_pipeline_etl.ipynb\n"
    "-- Sentencias en orden de ejecución. Ejecutarlo de nuevo reproduce las mismas tablas.\n\n"
)
cuerpo = ";\n\n".join(s.rstrip().rstrip(";") for s in PIPELINE) + ";\n"
(DIR_SQL / "pipeline_etl.sql").write_text(encabezado + cuerpo, encoding="utf-8")
print(f"{len(PIPELINE)} sentencias guardadas en {DIR_SQL / 'pipeline_etl.sql'}")

# Tablas finales en ecommerce.duckdb
ver("""
    SELECT table_name AS tabla, estimated_size AS registros_aprox, column_count AS columnas
    FROM duckdb_tables() WHERE NOT temporary
    ORDER BY regexp_extract(table_name, '_(raw|stg|clean|cuarentena)$', 1), table_name
""")

356 sentencias guardadas en d:\Users\luism\Documents\Personal\Uni\Maestria\analisis -de-datos\proyecto-inicial\sql\pipeline_etl.sql


,tabla,registros_aprox,columnas
0,catalogo_categorias,74,3
1,comparacion_antes_despues,42,10
2,conciliacion_pagos,99340,10
3,errores_inyectados,40,4
4,limites_iqr,3,7
5,quality_report,85,8
6,validacion_final,51,4
7,clientes_clean,99441,9
8,items_clean,112379,10
9,pagos_clean,103753,9
